In [ ]:
# Install dependencies
!pip uninstall -y -q torch torchvision torchaudio cudf easyocr numba-cuda dask-cuda pylibraft-cu12 raft-dask-cu12 pylibcudf-cu12 rmm-cu12 quantecon cuvs-cu12 ucxx-cu12 cuml-cu12 cuda-python cuml libcuml libcuvs libraft numba ydata-profiling google-colab
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch>=2.1.0 transformers>=5.15.0 datasets>=2.14.0 numpy>=1.24.0 matplotlib>=3.7.0 scipy>=1.10.0 papermill>=2.7.0 testbook>=0.4.2 diptest>=0.11.0
!pip install -q flash-linear-attention==0.4.2 --no-build-isolation
!pip install -q transformers==5.15.1


In [ ]:
import os
import sys
import time
import traceback
import torch
# Fix allocator fragmentation (exp16r OOM: reserved-but-unallocated
# grew to 1.1GB across alphas) — set BEFORE any CUDA use.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# KAGGLEHUB_CACHE: persist models across sessions
os.environ["KAGGLEHUB_CACHE"] = "/kaggle/working/.model_cache"
os.makedirs("/kaggle/working/.model_cache", exist_ok=True)
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.expanduser("~/.kaggle_test")
sys.path.insert(0, WORKING)
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    device_L, device_S = "cuda:0", "cuda:1"
    dtype_L = dtype_S = torch.float32
elif num_gpus == 1:
    device_L = device_S = "cuda:0"
    dtype_L = torch.float16
    dtype_S = torch.float32
else:
    device_L = device_S = "cpu"
    dtype_L = dtype_S = torch.float32
if num_gpus > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__} | L={device_L}, S={device_S}")
RES = os.path.join(WORKING, "results")
os.makedirs(RES, exist_ok=True)
# Resolve models: kernel_sources (prev commit outputs) -> working -> HF
def resolve_model(hf_id):
    model_dir = "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b"
    # 1. Check each slug in /kaggle/input/ — listdir only (isdir lies on mounts)
    input_dir = "/kaggle/input"
    try:
        slugs = os.listdir(input_dir)
        print(f"resolve {hf_id}: slugs={slugs}")
        for slug in slugs:
            p_sub = os.path.join(input_dir, slug)
            try:
                contents = os.listdir(p_sub)
            except OSError:
                continue
            if model_dir in contents:
                p = os.path.join(p_sub, model_dir)
                print(f"Found {hf_id} at {p} (via {slug})")
                return p
    except Exception as e:
        print(f"DEBUG: /kaggle/input/ check failed: {e}")
    # 2. Check WORKING
    wf = os.path.join(WORKING, model_dir)
    if os.path.isdir(wf):
        print(f"Found {hf_id} at {wf}")
        return wf
    # 3. Return HF ID - will download + persist
    print(f"Need to download {hf_id} from HF")
    return hf_id
# Load + save for persistence (calls from_pretrained, then save_pretrained if from HF)
def load_model(cls, hf_id, **kw):
    path = resolve_model(hf_id)
    model = cls.from_pretrained(path, **kw)
    # Persist if loaded from HF
    if path == hf_id:
        dst = os.path.join(WORKING, "qwen35-08b" if "0.8B" in hf_id else "qwen35-4b")
        if not os.path.isdir(dst):
            print(f"Saving {hf_id} to {dst}...")
            model.save_pretrained(dst)
            ensure_tokenizer(hf_id, dst)
            print(f"Saved")
    return model
# Ensure tokenizer files exist in a persisted model dir
def ensure_tokenizer(hf_id, dst):
    import transformers
    if not os.path.exists(os.path.join(dst, "tokenizer.json")):
        print(f"Saving tokenizer for {hf_id}...")
        transformers.AutoTokenizer.from_pretrained(hf_id).save_pretrained(dst)
SMALL = resolve_model("Qwen/Qwen3.5-0.8B")
LARGE = resolve_model("Qwen/Qwen3.5-4B")


In [ ]:
import contextlib
import logging
import torch
from transformers import DynamicCache
from transformers.cache_utils import (  # type: ignore[import-not-found]
    DynamicLayer,
    LinearAttentionLayer,
)
def compute_jacobian_for_tokens(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    prefix_cache=None,
    h_override: torch.Tensor | None = None,
) -> torch.Tensor:
    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.
    Args:
        model: Qwen3_5ForCausalLM instance.
        layer_idx: Layer index (0-based) to compute Jacobian at.
        input_ids: [1, S] token IDs.
        attention_mask: [1, S] attention mask.
        token_set: list of token IDs to compute gradients for.
        prefix_cache: optional pre-computed prefix cache (K/V for positions 0..S-2).
        h_override: if given, used as the hidden state of the last position at
            layer_idx INSTEAD of extracting it from a full forward. The prefix
            cache is still built from input_ids[:, :-1] (context unchanged); the
            caller is responsible for h_override being consistent with
            input_ids (e.g. h_override from the same sequence). Shapes: [1, 1, D],
            [1, D] or [D]. Device must match input_ids.device (no auto-move).
            dtype is cast to the model dtype. The tensor is cloned and detached
            (fresh leaf; the caller's tensor is never mutated).
    Returns:
        J: [K, D] Jacobian matrix (K = len(token_set), unnormalized).
    Interaction with compute_hidden_jacobian:
        compute_jacobian_for_tokens takes `h_override`; compute_hidden_jacobian
        takes `h`. Both mean "use this h instead of computing it from the
        forward", but the names differ for historical reasons (compute_hidden_
        jacobian already had `h`). With h_override=None the behavior is
        bit-identical to pre-v6.0.
    """
    model.eval()
    S = input_ids.shape[1]
    # Constraint: single sequence, no padding. The replay phase builds causal
    # masks from the cache length with attention_mask=None, so any padding
    # token in the prefix would be silently treated as valid context.
    if attention_mask is not None and not bool(attention_mask.all()):
        raise ValueError(
            "compute_jacobian_for_tokens: only unpadded single sequences are "
            "supported (the replay phase cannot honor a padding mask)."
        )
    if prefix_cache is None:
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    if h_override is None:
        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
        h = h.to(next(model.parameters()).dtype).detach().requires_grad_(True)
    else:
        h = _normalize_h_override(h_override, model, input_ids)
    # Re-enable grad: the decoder forward must build a graph over the leaf
    # h AND the row loop below must run autograd.grad with grad mode ON,
    # even when the caller is inside torch.no_grad() (e.g. eval loops in
    # exp18/19). A no_grad caller otherwise gets "element 0 ... does not
    # have a grad_fn" — the rows loop is part of the same enable_grad block.
    with torch.enable_grad():
        logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)
        J = _compute_jacobian_rows(logits, h, token_set)
    return J
def _normalize_h_override(
    h_override: torch.Tensor, model, input_ids: torch.Tensor
) -> torch.Tensor:
    """Validate + normalize h_override to a detached [1, 1, D] leaf.
    Checks (in order, before any compute): shape reducible to [1, 1, D],
    D == model hidden_size, device == input_ids.device. dtype is cast to the
    model dtype. The tensor is cloned+detached (fresh leaf; caller's tensor
    never mutated, no parasitic CopyBackwards nodes).
    """
    if h_override.dim() not in (1, 2, 3) or h_override.numel() == 0:
        raise ValueError(
            f"h_override: shape {tuple(h_override.shape)} is not reducible "
            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"
        )
    if any(s != 1 for s in h_override.shape[:-1]):
        raise ValueError(
            f"h_override: shape {tuple(h_override.shape)} is not reducible "
            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"
        )
    h = h_override.reshape(1, 1, -1)
    # D lookup: flat CausalLM has config.hidden_size; the multimodal wrapper
    # (Qwen3_5ForConditionalGeneration) exposes it on config.text_config only
    # (the outer config is a heterogeneity spec without hidden_size).
    D_expected = None
    cfg = model.config
    for cand in (getattr(cfg, "text_config", None), cfg):
        if cand is not None and hasattr(cand, "hidden_size"):
            D_expected = cand.hidden_size
            break
    if D_expected is None:
        raise ValueError(
            "h_override: cannot resolve model hidden_size "
            "(neither config nor config.text_config has it)"
        )
    D = h.shape[-1]
    if D != D_expected:
        raise ValueError(
            f"h_override: D={D} does not match model hidden_size={D_expected}"
        )
    if h.device != input_ids.device:
        raise ValueError(
            f"h_override: device {h.device} does not match "
            f"input_ids.device {input_ids.device} (no auto-move)"
        )
    model_dtype = next(model.parameters()).dtype
    # clone first, then cast: detach/clone act on the caller's values before
    # any cast, and `to(same dtype)` is a no-op, so the typical fp32 case
    # allocates exactly once (audit docs/changed_review.md, item 3).
    return h.detach().clone().to(model_dtype).requires_grad_(True)
def _run_prefix_pass(
    model, input_ids: torch.Tensor, attention_mask: torch.Tensor
) -> DynamicCache:
    """Prefix pass: all tokens except last -> cache."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids[:, :-1],
            attention_mask=attention_mask[:, :-1],
            use_cache=True,
            output_hidden_states=False,
        )
    return outputs.past_key_values
def _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:
    """Full pass: extract hidden state at layer_idx for last token."""
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
    # hidden_states[0] = embeddings, [l+1] = output of layer l
    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]
    return h  # [1, 1, D]
def _deep_clone_cache(cache):
    """Deep clone cache, ensuring all tensors are independent copies.
    Must copy every per-layer state field, not just the obvious tensors. Both
    layer types swallow unknown constructor kwargs, so passing
    ``keys=`` / ``conv_states=`` / ``recurrent_states=`` silently discards the
    state and yields an empty cache. We therefore assign attributes directly:
    * DynamicLayer (full attention): keys / values / is_initialized / dtype /
      device. Without this, ``update()`` lazy-initialises an empty cache and
      each layer attends only to the current token.
    * LinearAttentionLayer (GDN): conv_states / recurrent_states plus the flag
      dicts (``has_previous_state`` etc.). Without these, GDN layers treat the
      new token as the first in the sequence and drop all history.
    All cloned state tensors are .detach()-ed: the prefix cache is pure data
    (built under no_grad). Detaching drops pointless CopyBackwards graph nodes
    in the enable_grad decode. The transformers 5.15 in-place cache updates
    (update_recurrent_state's copy_, causal_conv1d_update's state update —
    cudagraph static-address optimisation) are NOT fixed by the detach —
    the in-place copy_ lands on the kernel's saved input regardless. Those
    are handled by _clone_gdn_states_on_entry (clone at kernel entry so the
    read object and the in-place-written object are different objects).
    Extra unknown attributes are deep-copied as a forward-compat safety net.
    Cache OBJECT-level attributes (e.g. ``offloading`` on 5.15.1; 4.x carried
    ``_seen_tokens``) are copied from ``vars(cache)`` after the layer loop:
    a fresh ``DynamicCache()`` would reset them. On 5.15.1 ``get_seq_length()``
    is derived from ``keys.shape`` (not an object field), so the post-clone
    assert below passes on both paths — on a version that tracks sequence
    length in cache state it turns a silent Jacobian corruption into a loud
    error.
    ``copy.deepcopy(prefix_cache)`` DOES produce independent copies on 5.15.1
    (tensor values are not shared). _deep_clone_cache exists for (a) detached
    tensors (no spurious CopyBackwards in the enable_grad decode) and
    (b) robustness to per-layer attribute renames — not because deepcopy
    shares storage. Neither clone fixes the in-place write into a graph-saved
    node; that is _clone_gdn_states_on_entry's job (see its docstring).
    """
    import copy as _copy
    def _cl(t):
        return t.clone().detach() if t is not None else None
    new_cache = DynamicCache()
    for idx in range(len(cache.layers)):
        layer = cache.layers[idx]
        if isinstance(layer, DynamicLayer):
            # Deliberately NOT a vars()-walk: full-attention state fields are
            # stable across versions, and a rename would fail LOUD
            # (AttributeError), not silently corrupt the decode. The GDN
            # branch below IS generic, because GDN state fields have
            # demonstrably drifted between versions/backends.
            nl = DynamicLayer()
            nl.keys = _cl(layer.keys)
            nl.values = _cl(layer.values)
            nl.is_initialized = layer.is_initialized
            nl.dtype = getattr(layer, "dtype", None)
            nl.device = getattr(layer, "device", None)
            for k, v in vars(layer).items():
                if k not in vars(nl):
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
        else:
            # GDN (LinearAttentionLayer). Clone every state field dynamically
            # rather than hardcoding transformers-5.15.1 attribute names
            # (number_of_states, conv_states, ...): other transformers versions
            # on other backends (e.g. the Kaggle TPU torch 2.8 image) name some
            # differently, and a hardcoded name crashes with AttributeError.
            # Tensors (or dicts of tensors) are clone+detached; scalars and
            # flag-dicts are deep-copied. This is the generic form of the old
            # field-by-field copy and behaves identically on 5.15.1.
            nl = LinearAttentionLayer()
            for k, v in vars(layer).items():
                if isinstance(v, torch.Tensor):
                    setattr(nl, k, _cl(v))
                elif isinstance(v, dict) and v and all(
                    isinstance(x, torch.Tensor) for x in v.values()
                ):
                    setattr(nl, k, {kk: _cl(xv) for kk, xv in v.items()})
                else:
                    setattr(nl, k, _copy.deepcopy(v))
            new_cache.layers.append(nl)
    # Cache object-level state (offloading flag on 5.15.1; 4.x: _seen_tokens):
    # a fresh DynamicCache() would reset it. ``layers`` is rebuilt above;
    # list-valued attrs are skipped to avoid aliasing the original.
    for k, v in vars(cache).items():
        if k == "layers" or isinstance(v, list):
            continue
        setattr(new_cache, k, _copy.deepcopy(v))
    # Guard: the clone must report the same sequence length (loud, not
    # silent, on a version that tracks it in cache state).
    assert new_cache.get_seq_length() == cache.get_seq_length()
    return new_cache
@contextlib.contextmanager
def _clone_gdn_states_on_entry():
    """Clone GDN kernel state inputs at entry: read object != written object.
    transformers 5.15 Qwen3_5GatedDeltaNet.forward resolves the kernels as
    free variables of its own module (modeling_qwen3_5), so patching the
    module attributes works — verified on 5.15.1 (calls at modeling L457/
    L506). On
    a version where GDN forward binds the kernels by a different route
    (an alias imported into another module), the patch silently no-ops and
    the in-place autograd error below resurfaces LOUD — that is the
    detection signal; there is no silent degradation.
    The recurrent loop saves the read object (version 0) for backward.
    Afterwards update_recurrent_state does a stored.copy_(new) IN PLACE on
    the very same object (5.15 cudagraph static-address optimisation) →
    version 0→1 → "modified by an inplace operation" in any backward through
    the decode (create_graph not required; the clone is also a no-op safety
    net for create_graph=True graph retention). Cloning at kernel entry makes
    the loop's saved input the fresh clone; the in-place write lands on the
    cache's original (a different object) → no in-place on a saved node.
    Re-entrancy: nesting is a NO-OP (counter below) — the outer context's
    wrappers are already active and cloning is idempotent (clone-of-clone
    changes nothing). Overwriting mq.<kernel> from OUTSIDE during the patch
    window is user error and is NOT protected (the outer exit restores its
    captured orig, discarding the overwrite).
    If the Qwen3.5 module cannot be imported, a WARNING is logged ONCE per
    interpreter (not per decode; non-Qwen3.5 builds are fine — for Qwen3.5
    with enable_grad=True the in-place autograd error is NOT protected).
    """
    global _GDN_PATCH_DEPTH
    try:
        from transformers.models.qwen3_5 import modeling_qwen3_5 as mq
    except Exception:
        if not getattr(_clone_gdn_states_on_entry, "_warned", False):
            logging.getLogger(__name__).warning(
                "GDN monkey-patch skipped (transformers.models.qwen3_5 not "
                "importable). If the model is Qwen3.5 with enable_grad=True, "
                "in-place autograd errors may occur.")
            _clone_gdn_states_on_entry._warned = True
        yield  # non-Qwen3.5 build: nothing to patch
        return
    _GDN_PATCH_DEPTH += 1
    if _GDN_PATCH_DEPTH > 1:
        # Nested: outer context already patched; wrappers chain, and the
        # clone is idempotent, so nothing to do here.
        yield
        _GDN_PATCH_DEPTH -= 1
        return
    orig_rec = mq.torch_recurrent_gated_delta_rule
    orig_conv = mq.causal_conv1d_update
    def rec(q, k, v, g, beta, *args, initial_state=None, output_final_state=True,
            use_qk_l2norm_in_kernel=False, **kw):
        # initial_state/output_final_state arrive as KEYWORDS in 5.15.1
        # (and may be positional on other versions, hence *args); clone the
        # bound state, forward everything else untouched. A kernel whose
        # signature drops these names falls through to **kw and fails loud.
        if initial_state is not None:
            initial_state = initial_state.clone()
        return orig_rec(q, k, v, g, beta, *args, initial_state=initial_state,
                        output_final_state=output_final_state,
                        use_qk_l2norm_in_kernel=use_qk_l2norm_in_kernel, **kw)
    def conv(h, cs, *args, **kw):
        # (cs, weight, [bias, activation], ...) — positional tail is
        # version-dependent; forward everything we don't own.
        if cs is not None:
            cs = cs.clone()
        return orig_conv(h, cs, *args, **kw)
    mq.torch_recurrent_gated_delta_rule = rec
    mq.causal_conv1d_update = conv
    try:
        yield
    finally:
        mq.torch_recurrent_gated_delta_rule = orig_rec
        mq.causal_conv1d_update = orig_conv
        _GDN_PATCH_DEPTH -= 1
_GDN_PATCH_DEPTH = 0
def _run_decoder_pass(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers using prefix cache."""
    # Clone cache tensors independently to avoid in-place mutation
    cache_clone = _deep_clone_cache(prefix_cache)
    # Wrapper-safe: flat CausalLM -> model.model; mm wrapper -> .language_model
    text_model = _lm_module(model)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    layer_types = _layer_types(model)
    n_layers = _n_layers(model)
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    x = text_model.norm(x)
    logits = model.lm_head(x[:, -1, :])
    return logits
def _build_causal_masks(text_model, text_pos, cache, x):
    """Build causal mask mapping for a single token with cache."""
    from transformers.masking_utils import (
        create_causal_mask,
        create_recurrent_attention_mask,
    )
    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)
    mask_kwargs = {
        "config": text_model.config,
        "inputs_embeds": dummy_embeds,
        "attention_mask": None,
        "past_key_values": cache,
        "position_ids": text_pos,
    }
    return {
        "full_attention": create_causal_mask(**mask_kwargs),
        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),
    }
def _compute_jacobian_rows(
    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]
) -> torch.Tensor:
    """Compute Jacobian rows via ONE batched backward (is_grads_batched).
    Replaces the per-row autograd.grad loop (K backwards through the
    decoder tail) with a single vmap-style pass: grad_outputs=eye(K)
    gives all K rows at once, 3-15x faster on T4 (review point 5).
    Verified identical to the loop on synthetic linear + nonlinear graphs;
    on the real GDN decoder tail this is an EMPIRICAL property of the
    transformers 5.15 kernels (torch_recurrent_gated_delta_rule / causal
    conv1d update under vmap-style batched backward) — there is no
    per-row fallback, so a transformers upgrade that breaks this shows up
    as a runtime error, not silent degradation.
    """
    K = len(token_set)
    return torch.autograd.grad(
        outputs=logits[0, token_set],
        inputs=h,
        grad_outputs=torch.eye(K, device=h.device, dtype=h.dtype),
        is_grads_batched=True,
        retain_graph=False,
    )[0][:, 0, 0, :]
def softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    """Gauge-invariant Jacobian: J^(p) = (diag(p) - pp^T) @ J.
    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k
    Removes the common-mode shift (gauge) from raw Jacobian.
    """
    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]
    return p[:, None] * (J - J_bar)  # [K, D]
def uniform_centered(J: torch.Tensor) -> torch.Tensor:
    """Uniform centering: J - mean(J). Removes average shift."""
    return J - J.mean(dim=0, keepdim=True)
def _last_layer_jacobian(h: torch.Tensor, norm: torch.nn.Module) -> torch.Tensor:
    """Analytical J = d h_norm / d h at the final norm (no backward passes).
    Both Qwen3_5RMSNorm and nn.LayerNorm are handled. J is evaluated at the
    norm INPUT (h), never at h_norm — evaluating the formula at the norm
    output gives a matrix with ~0.87 relative error (no longer LayerNorm-like
    statistics). Qwen3.5's final norm is Qwen3_5RMSNorm:
      h_norm = (h * rsqrt(mean(h^2) + eps) * (1 + w))
      J = r · diag(1+w) · (I - xhat xhat^T / D),  r = rsqrt(mean(h^2)+eps), xhat = r*h
    LayerNorm: J = (I - 1/D - xhat xhat^T / D) / std, rows scaled by w.
    """
    # h may arrive as [1, S, D] (full-sequence hidden_states) or [1, D]/[D].
    # The last-layer READOUT Jacobian is the FINAL-position map (it feeds the
    # final-token logits), so collapse a [1, S, D] to its last position first.
    # BUG this fixes: [1, S, D].reshape(1,-1) -> [1, S*D] -> [S*D, S*D] identity
    # (e.g. [65536, 65536] = 16 GB) -> OOM on GPU / XLA compile-abort on TPU
    # (previously misattributed to an "XLA phantom dot" shape bug).
    if h.dim() == 3:
        h = h[:, -1, :]
    x = h.reshape(1, -1).float()  # [1, D], the norm INPUT (row vector)
    D = x.shape[-1]
    eps = float(getattr(norm, "eps", None) or 1e-5)
    w = getattr(norm, "weight", None)
    dev = x.device
    if getattr(norm, "bias", None) is not None:  # LayerNorm family
        mu = x.mean(dim=-1, keepdim=True)
        std = (x.var(dim=-1, unbiased=False, keepdim=True) + eps).sqrt()
        xhat = (x - mu) / std  # [1, D]
        J = (
            torch.eye(D, device=dev)
            - torch.ones(D, D, device=dev) / D
            - xhat.T @ xhat / D
        ) / std.squeeze(0)
        if w is not None:
            J = J * w.float().squeeze(0).unsqueeze(1)  # J = diag(w) @ J
        return J
    # RMSNorm family (Qwen3_5RMSNorm, Llama-style).
    r = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    xhat = x * r  # [D]
    # y_j = (1+w_j) r x_j; J = r * diag(1+w) @ (I - xhat xhat^T / D)
    # (derived: J_ij = g_j (r d_ij - r^3 x_i x_j / D) = g_j r (d_ij - xhat_i xhat_j / D))
    g = 1.0 + w.float().squeeze(0) if w is not None else None
    M = torch.eye(D, device=dev) - xhat.T @ xhat / D
    if g is None:
        return r.squeeze(0) * M
    return r.squeeze(0) * torch.diag(g) @ M
def compute_hidden_jacobian(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> torch.Tensor:
    """Compute Jacobian of normalized last hidden state w.r.t. hidden state at layer_idx.
    J_norm = d(h_norm) / d(h_{layer_idx}), shape [D, D].
    Independent of lm_head — computed once, reused across W updates.
    Then J_S = lm_head.weight[token_set] @ J_norm is differentiable through W.
    For last_layer: J_norm is LayerNorm Jacobian (analytical, no backward passes).
    Args:
        create_graph: if True, J_norm has grad_fn.
            - If h is None: h is extracted from a full forward and stays
              connected to model params (full gradient flow, as before).
            - If h is given: h is used AS-IS instead of the forward (the
              forward is skipped). A ready leaf h (requires_grad=True,
              grad_fn=None) is used IN PLACE — its .grad receives the
              backpropagated derivative d J_norm/d h. J_norm still depends
              on the decoder WEIGHTS between layer_idx and the final norm
              (the decode pass), so gradients also reach those params; only
              the params of layers <= layer_idx / embeddings are excluded.
                      Costs ~D backward passes per sample.
        verbose: print progress during J_norm computation.
        h: pre-computed hidden state at layer_idx (reuse from a prior forward,
            e.g. KL-forward in exp17). Respected in BOTH create_graph paths
            (v6.0 bug-fix: the create_graph=True path previously ignored it
            and unconditionally re-extracted h from a full forward). Not
            renamed to h_override for backward compatibility; the asymmetry
            with compute_jacobian_for_tokens(h_override=...) is intentional.
    """
    model.eval()
    n_layers = _n_layers(model)
    if create_graph:
        # Keep gradient graph alive: h is connected to model params (+ any hooks)
        S = input_ids.shape[1]
        # Build prefix cache FIRST — before any forward involving hooks.
        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
        if h is None:
            # Full forward to get h at layer_idx — connected to all model params + hooks
            with torch.enable_grad():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
            assert outputs.hidden_states is not None  # requested
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D], connected!
        # Normalize [1, D] / [D] -> [1, 1, D]; keep any existing grad graph intact.
        h = h.reshape(1, 1, -1) if h.dim() in (1, 2) else h
        model_dtype = next(model.parameters()).dtype
        assert h is not None  # narrowed: param was Tensor|None
        h = h.to(model_dtype)  # preserve grad graph
        if h.grad_fn is None:
            if h.requires_grad:
                # Caller passed a ready leaf that already requires grad
                # (e.g. a representation-holonomy h_leaf). Use it IN PLACE:
                # detach() would create a NEW leaf and the caller's tensor
                # would never receive .grad. It is already a leaf (grad_fn
                # None), so no graph surgery is needed.
                pass
            else:
                # Fully frozen graph: no trainable path reaches this layer
                # (e.g. linear_attention pair with no LoRA patch, or a
                # no_grad caller), OR a disconnected h was passed in. Make h
                # a leaf so the decoder pass below still yields a
                # differentiable h_norm; J_norm values are identical to the
                # connected case.
                h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            with torch.enable_grad():
                h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            with torch.enable_grad():
                h_norm = _run_decoder_to_norm_compat(
                    model, prefix_cache, h, S, layer_idx
                )
        # [1, 1, D] — _compute_jacobian_rows_full indexes grad[0, 0, :].
        # enable_grad: autograd.grad needs grad mode ON even when the
        # caller is inside torch.no_grad().
        with torch.enable_grad():
            J_norm = _compute_jacobian_rows_full(
                h_norm, h, create_graph=True, verbose=verbose
            )
        return J_norm
    # Last layer: analytical norm Jacobian, no backward passes.
    if layer_idx == n_layers - 1:
        with torch.no_grad():
            if h is None:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )
                assert outputs.hidden_states is not None  # requested
                h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
            assert h is not None
            h = h.to(next(model.parameters()).dtype)
        return _last_layer_jacobian(h, _lm_module(model).norm)
    # General case: decode through remaining layers + norm
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    if h is None:
        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)
    model_dtype = next(model.parameters()).dtype
    h = h.to(model_dtype).detach().requires_grad_(True)
    # Re-enable grad: h_norm must carry a grad_fn over h even if the caller
    # is inside torch.no_grad() (e.g. eval loops in exp16r/exp17). The row
    # loop below (autograd.grad) ALSO needs grad mode ON — a no_grad caller
    # otherwise gets "element 0 ... does not have a grad_fn".
    with torch.enable_grad():
        h_norm = _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
        J_norm = _compute_jacobian_rows_full(
            h_norm, h, create_graph=create_graph, verbose=verbose
        )
    return J_norm
def _run_decoder_to_norm(
    text_model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """Decode h through remaining layers, return norm(x) without lm_head."""
    cache_clone = _deep_clone_cache(prefix_cache)
    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)
    text_position_ids = pos[0]
    rope_position_ids = pos[1:]
    position_embeddings = text_model.rotary_emb(h, rope_position_ids)
    x = h
    n_layers = text_model.config.num_hidden_layers
    start = layer_idx + 1
    causal_mask_mapping = _build_causal_masks(
        text_model, text_position_ids, cache_clone, x
    )
    with _clone_gdn_states_on_entry():
        for i in range(start, n_layers):
            layer_type = text_model.config.layer_types[i]
            attn_mask = causal_mask_mapping[layer_type]
            x = text_model.layers[i](
                x,
                position_embeddings=position_embeddings,
                attention_mask=attn_mask,
                position_ids=text_position_ids,
                past_key_values=cache_clone,
            )
    return text_model.norm(x[:, -1, :])
def _compute_jacobian_rows_full(
    output: torch.Tensor,
    h: torch.Tensor,
    create_graph: bool = False,
    verbose: bool = False,
) -> torch.Tensor:
    """Full [D, D] Jacobian via autograd.grad."""
    import time as _time
    D = h.shape[-1]
    out_flat = output.flatten()
    grads = []
    # create_graph=True: every row must retain the graph — the caller
    # backpropagates a final loss through J_norm afterwards (exp17 bridge
    # loss -> o_proj HVP). retain=(i<D-1) would free the (second-order)
    # graph after the last row, so the caller's first backward would fail
    # with "backward through the graph a second time". Memory is bounded
    # by the refresh schedule (one graph per si, freed at the next refresh
    # / pair end).
    start = _time.time()
    for i in range(D):
        grad = torch.autograd.grad(
            outputs=out_flat[i],
            inputs=h,
            retain_graph=(True if create_graph else i < D - 1),
            create_graph=create_graph,
        )[0]
        grads.append(grad[0, 0, :])
        if verbose and i % 128 == 0:
            elapsed = _time.time() - start
            rate = i / elapsed if elapsed > 0 else 999
            eta = elapsed * (D / i) - elapsed if i > 0 else 0.0
            print(f"        J_norm: {i}/{D} ({rate:.0f}/s, eta={eta:.0f}s)")
    J = torch.stack(grads)
    elapsed_total = _time.time() - start
    if verbose:
        print(f"        J_norm done: {D}/{D} in {elapsed_total:.1f}s")
    return J
def _lm_module(model):
    """The text model with .layers/.norm/.rotary_emb.
    Flat CausalLM: model.model IS the text model. Multimodal wrapper
    (Qwen3_5ForConditionalGeneration): model.model.language_model.
    """
    mm = model.model
    if hasattr(mm, "layers"):
        return mm
    return mm.language_model
def _n_layers(model) -> int:
    """num_hidden_layers for flat or wrapper configs."""
    cfg = model.config
    return int(
        cfg.num_hidden_layers
        if hasattr(cfg, "num_hidden_layers")
        else cfg.text_config.num_hidden_layers
    )
def _layer_types(model) -> list:
    """layer_types for flat or wrapper configs."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
def _run_decoder_to_norm_compat(
    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int
) -> torch.Tensor:
    """_run_decoder_to_norm over _lm_module (flat or nested model)."""
    return _run_decoder_to_norm(_lm_module(model), prefix_cache, h, S, layer_idx)
def h_norm_at_perturbed_h(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    h: torch.Tensor,
) -> torch.Tensor:
    """h_norm after the prefix, with the l_S hidden state pinned to `h`.
    Finite-diff support (B2 spot check): call twice, h and h + eps*e_j,
    and difference the two h_norms. h shape [1, 1, D] (any device/dtype);
    the pinning is done on a copy — the input is not mutated.
    """
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.no_grad():
        return _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)
def _forward_h_norm(
    model,
    layer_idx,
    input_ids,
    attention_mask,
    h: torch.Tensor | None = None,
):
    """Forward once -> (h, h_norm), h graph-connected at layer l_S.
    Shared by compute_hidden_jacobian_rows and bridge_rows_backward (A4):
    the prefix pass runs before the model forward (hook-safe), h is the
    l_S hidden state (kept in graph so a caller that patched o_proj / lm_head
    gets HVP/bridge grads from the downstream backward), h_norm = RMSNorm(h)
    (last layer) or decoder-to-norm (otherwise).
    h (optional): reuse a pre-computed l_S hidden state instead of doing a
    fresh full forward (v23.1: exp17's KL forward already produced it, so
    the refresh reuses it and only runs the cheap decoder-tail-to-norm).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)
    with torch.enable_grad():
        if h is None:
            # standalone: full forward to obtain h (original A4 behavior)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )
            assert outputs.hidden_states is not None  # requested
            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]
        assert h is not None
        h = h.to(next(model.parameters()).dtype)
        if h.grad_fn is None:
            h = h.detach().requires_grad_(True)
        if layer_idx == n_layers - 1:
            h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]
        else:
            h_norm = _run_decoder_to_norm_compat(
                model, prefix_cache, h, S, layer_idx
            )
    return h, h_norm
def bridge_rows_backward(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    J_tgt_p: torch.Tensor,
    extra_loss: torch.Tensor | None = None,
    chunk_size: int = 8,
    verbose: bool = False,
    h: torch.Tensor | None = None,
) -> tuple[torch.Tensor, float]:
    """v23 A4: chunked bridge backward (the v22 OOM fix).
    Forward once + J_bar once, then build each chunk of K rows and backprop
    its (1 - cos) bridge loss immediately (extra_loss — the KL — is
    backproped with the first chunk). Backproping per chunk frees that
    chunk's 2nd-order row-graphs at once, so peak = chunk_size + 1 (the
    shared J_bar) second-order graphs, not K. Per-chunk gradients are
    identical to a single backward on the total (1 - cos).mean() loss.
    Gradients reach W_rows (live lm_head + B_lh @ A_lh) and o_proj (HVP
    through h). Returns (J_S_p [K, D] detached, bridge_loss scalar).
    h (optional): reuse a pre-computed l_S hidden state (v23.1) so this
    does NOT redo the full forward — only the cheap decoder-tail-to-norm
    runs. exp17's KL forward already produced the same h.
    """
    import time as _time
    K = len(token_set)
    h, h_norm = _forward_h_norm(
        model, layer_idx, input_ids, attention_mask, h=h
    )
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=True,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # shared; freed by the last chunk's bwd
    # extra_loss (the KL) gets a SEPARATE backward before the chunk loop.
    # retain_graph=True keeps the shared h-graph alive for the chunks while
    # the KL's OWN forward graph (a separate forward in exp17) frees here.
    # We deliberately do NOT add it to chunk 0: in a shared-forward test/
    # wiring that would free the h-graph the chunks still need.
    if extra_loss is not None:
        extra_loss.backward(retain_graph=True)
    chunks: list[torch.Tensor] = []
    total_bridge = 0.0
    for c in range(0, K, chunk_size):
        hi = min(c + chunk_size, K)
        is_last = (c + chunk_size) >= K
        rows_c = torch.stack(
            [p[i] * (_row_grad(W_rows[i]) - J_bar) for i in range(c, hi)]
        )
        tgt_c = J_tgt_p[c:hi].float()
        cos_c = torch.nn.functional.cosine_similarity(rows_c.float(), tgt_c, dim=1)
        chunk_loss = (1.0 - cos_c).sum() / K
        # retain_graph on all-but-last keeps the ONE shared h forward graph
        # alive so the next chunk can still build its rows; the chunk's
        # 2nd-order row-graph is released by refcounting once we del it
        # (forward nodes stay, referenced by h / h_norm / J_bar).
        chunk_loss.backward(retain_graph=not is_last)
        total_bridge += chunk_loss.item()
        chunks.append(rows_c.detach())
        del rows_c, cos_c, chunk_loss, tgt_c
        if verbose:
            print(
                f"        bridge_rows: {hi}/{K} "
                f"({hi / max(_time.time() - start, 1e-6):.1f} chunks/s)"
            )
    return torch.cat(chunks), total_bridge
def compute_hidden_jacobian_rows(
    model,
    layer_idx: int,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    token_set: list[int],
    p: torch.Tensor,
    W_rows: torch.Tensor,
    create_graph: bool = False,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """v23 A1: per-row J_S_p = p_i*(J_i - J_bar), no dense [D,D] J_norm.
    K-subset softmax gauge — the SAME gauge as the exp16 target (p_s =
    softmax(full_logits[token_set])) and the exp17 dense bridge
    (softmax_jacobian(W[token_set]@J_norm, p_L)):
      p = re-normalized dist over the K-subset [K], W_rows = W[token_set]
      [K, D]; J_bar = d(h_norm @ (p @ W_rows))/dh; row_i =
      p[i] * (d(h_norm @ W_rows[i])/dh - J_bar).
    h stays graph-connected (via _run_decoder_to_norm_compat, which
    routes through _lm_module so both flat CausalLM and the multimodal
    wrapper work), so a caller that patched the target layer's o_proj
    and/or lm_head gets its HVP/bridge gradients from the downstream
    loss backward. chunk_size (A4) bounds the 2nd-order graphs alive
    at once; create_graph=True retains them for a later loss backward,
    create_graph=False frees each chunk after the gradient (for cheap
    no-grad refreshes / evals).
    """
    model.eval()
    n_layers = _n_layers(model)
    S = input_ids.shape[1]
    h, h_norm = _forward_h_norm(model, layer_idx, input_ids, attention_mask)
    return _compute_jacobian_rows_matrixfree(
        h_norm,
        h,
        W_rows,
        p,
        token_set,
        create_graph=create_graph,
        chunk_size=chunk_size,
        verbose=verbose,
    )
def _compute_jacobian_rows_matrixfree(
    h_norm: torch.Tensor,
    h: torch.Tensor,
    W_rows: torch.Tensor,
    p: torch.Tensor,
    token_set: list[int],
    create_graph: bool = True,
    chunk_size: int = 8,
    verbose: bool = False,
) -> torch.Tensor:
    """J_S_p [K, D] rows, K-subset softmax gauge, no dense [D,D] J_norm.
    p [K] = re-normalized softmax over the K-subset (exp16/exp17 gauge),
    W_rows [K, D] = the selected rows. J_bar = d(h_norm @ (p @ W_rows))/dh;
    row_i = p[i]*(d(h_norm @ W_rows[i])/dh - J_bar). Identical to
    softmax_jacobian(W_rows @ J_norm, p) but with K backward passes
    instead of a retained [D,D] 2nd-order matrix (v22 OOM).
    Chunked (A4): only chunk_size row-graphs alive at once.
    """
    import time as _time
    K = len(token_set)
    def _row_grad(w: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(
            outputs=h_norm @ w,
            inputs=h,
            retain_graph=True,
            create_graph=create_graph,
        )[0][0, 0, :]
    start = _time.time()
    J_bar = _row_grad(p @ W_rows)  # [D]; p@W_rows = effective selected row
    rows = []
    for c in range(0, K, chunk_size):
        for local, i in enumerate(range(c, min(c + chunk_size, K))):
            row = p[i] * (_row_grad(W_rows[i]) - J_bar)
            rows.append(row)
        if verbose:
            done = min(c + chunk_size, K)
            el = _time.time() - start
            print(f"        J_S_p: {done}/{K} ({done / max(el, 1e-6):.1f}/s)")
    out = torch.stack(rows)
    elapsed = _time.time() - start
    if verbose:
        print(f"        J_S_p done: {K}/{K} in {elapsed:.1f}s")
    return out


In [ ]:
import hashlib
import os
import torch
_gramian_cache: dict[str, torch.Tensor] = {}
_GRAMIAN_DIR = os.path.join("results", ".gramians")
def jacobian_gap(
    J_L: torch.Tensor,
    J_S: torch.Tensor,
    E_L: torch.Tensor,
    E_S: torch.Tensor,
    probs_L: torch.Tensor,
    threshold: float = 0.0,
) -> float:
    """Compute the Jacobian gap (alignment loss, LOWER = BETTER) between large and small models.
    Terminology (v4.0): gap = weighted (1 - cos), lower=better; bridge = weighted
    cos, higher=better; gap = 1 - bridge at equal weights. Callers reporting
    "bridge" (E-16a/b) invert: bridge = 1 - jacobian_gap(...). The legacy
    `jacobian_bridge` alias (which silently returned the GAP direction despite
    its name) was removed after being identified as an inversion trap — the
    inversion lives in the experiment kernels (kaggle_kernel/kernel-exp1*/...py),
    not here.
    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.
    The inverse Gramian (E_S^T @ E_S)^{-1} (disk-cached, see _inverse_gramian)
    corrects for non-orthogonal embeddings, so identity (J_L == J_S,
    E_L == E_S) yields gap ~0.
    threshold: row mask on BOTH norms, (n1 > threshold) & (n2 > threshold).
    Default 0.0 = sentinel-only mode: only exactly-zero rows are masked
    (no decay filtering). Historical default was 1e-6; the paper's
    relative quantile tau_l = 0.01 * median_i ||J_S[i]|| (E-6 calibration,
    ~1e-4 at L18) can be passed explicitly to enable decay masking.
    """
    G_inv = _inverse_gramian(E_S)
    J_L_proj = _project_to_vocab(J_L, E_L)
    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)
    mask = _valid_mask(J_S, J_S_target, threshold)
    if mask.sum() == 0:
        return 1.0
    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)
def _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:
    """Project J_L [K, D_L] through E_L^T -> [K, V]."""
    return J_L.float() @ E_L.float().T
def _project_to_small(
    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor
) -> torch.Tensor:
    """Project [K, V] through E_S @ G_inv -> [K, D_S].
    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.
    """
    return J_L_proj @ E_S.float() @ G_inv
def _inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""
    key = _gramian_key(E)
    if key not in _gramian_cache:
        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")
        if os.path.exists(path):
            _gramian_cache[key] = torch.load(path, weights_only=True)
        else:
            _gramian_cache[key] = _compute_inverse_gramian(E)
            os.makedirs(_GRAMIAN_DIR, exist_ok=True)
            torch.save(_gramian_cache[key], path)
    return _gramian_cache[key].to(E.device)
def _gramian_key(E: torch.Tensor) -> str:
    """Stable hash of an embedding matrix for disk caching.
    Hashes shape/device header + first, last, and 10 deterministic mid-rows
    (uniformly spaced, seeded from row count so the sample is stable across
    runs but covers the interior). First/last rows alone would collide for
    matrices that share boundary rows (e.g. PAD/UNK) but differ internally.
    Hashing the full 248k×1024 matrix would be prohibitively expensive.
    """
    V = E.shape[0]
    idx = [0, V - 1]
    if V > 4:
        step = (V - 1) // 10
        idx += sorted({(i * step) % (V - 1) + 1 for i in range(1, 10)})
    header = f"{tuple(E.shape)}|{str(E.device)}".encode()
    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()
    for i in sorted(set(idx)):
        data += E[i].cpu().numpy().tobytes()
    h = hashlib.sha256(data).hexdigest()[:16]
    return f"g_{h}"
def _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:
    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""
    G = E.float().T @ E.float()
    return torch.linalg.pinv(G)
def _valid_mask(
    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float
) -> torch.Tensor:
    """Mask rows where either norm <= threshold (sentinel rows).
    threshold=0.0 (default): only exactly-zero rows are masked.
    threshold>0: decay masking (absolute threshold; the paper's relative
    tau_l = 0.01*median is applied by the caller, E-6 calibration).
    """
    n1 = J_S.float().norm(dim=1)
    n2 = J_target.float().norm(dim=1)
    return (n1 > threshold) & (n2 > threshold)
def _weighted_cosine_loss(
    J_S: torch.Tensor,
    J_target: torch.Tensor,
    probs: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    """Weighted cosine dissimilarity loss."""
    cos_sim = torch.nn.functional.cosine_similarity(
        J_S.float()[mask], J_target[mask], dim=1
    )
    w = probs[mask]
    w = w / (w.sum() + 1e-8)
    return (w * (1.0 - cos_sim)).sum().item()


In [ ]:
import torch
def _validate(jacobians: list[torch.Tensor]) -> None:
    if not jacobians:
        raise ValueError("jacobians: empty list")
    ref = jacobians[0]
    for J in jacobians[1:]:
        if J.shape != ref.shape:
            raise ValueError(f"jacobians: shape mismatch {tuple(J.shape)} vs {tuple(ref.shape)}")
        if J.device != ref.device:
            raise ValueError(f"jacobians: device mismatch {J.device} vs {ref.device}")
def compute_esg(
    jacobians: list[torch.Tensor],
    n_components: int = 32,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute ESG from a list of Jacobian matrices.
    Pooling is INTRA-TRAJECTORY: all J_i must be positions of ONE sample
    (document). Cross-sample pooling is invalid — see module docstring /
    paper §15 (E-0.1); aggregate across samples at the scalar-metric level.
    Args:
        jacobians: list of N Jacobian matrices (one trajectory), each [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: [min(n_components, K)] sorted descending (K < n_components
        possible in degenerate cases; in practice K = |𝒮| ≥ 32).
        eigenvectors: [min(n_components, K), K] corresponding eigenvectors.
    """
    _validate(jacobians)
    G = _build_gramian(jacobians)
    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)
    return eigenvalues, eigenvectors
def compute_esg_per_sample(
    sample_jacobians: list[list[torch.Tensor]],
    n_components: int = 32,
) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
    """Sample-local ESG: one Gramian per sample, NO cross-sample pooling.
    Args:
        sample_jacobians: list over samples; each entry is a list of the
            sample's per-position Jacobians [K, D].
        n_components: number of top eigenvalues/vectors to return.
    Returns:
        eigenvalues: list [n_samples] of [n_components].
        eigenvectors: list [n_samples] of [n_components, K].
    Cross-sample aggregation happens on scalars (lambda, PR, SNR, r_eff),
    never on the matrices (E-0.1: cross-sample J's are near-orthogonal).
    """
    if not sample_jacobians:
        raise ValueError("sample_jacobians: empty list")
    eigenvalues = []
    eigenvectors = []
    for Js in sample_jacobians:
        ev, evec = compute_esg(Js, n_components=n_components)
        eigenvalues.append(ev)
        eigenvectors.append(evec)
    return eigenvalues, eigenvectors
def cross_positional_gramian(
    J_t: torch.Tensor,
    J_tp: torch.Tensor,
) -> torch.Tensor:
    """Cross-positional Gramian Δ_U(t, t') = J_t @ J_tp.T.
    Args:
        J_t:  [K, D] gauge-free Jacobian at position t.
        J_tp: [K, D] gauge-free Jacobian at position t'.
    Returns:
        [K, K] Gramian (symmetric iff t == t').
    Note:
        - NOT a pooling operation: each call uses exactly two positions.
          See module docstring for intra-trajectory pooling (compute_esg)
          and cross-sample pooling validity (E-0.1).
        - When t == t', result equals the inner-product part of
          compute_esg([J_t]) for N=1: compute_esg([J_t])[0][0] ==
          eigenvalue_0(Δ_U(t, t)). Checked in selfcheck.
        - Caller must ensure J_t, J_tp come from the same model / edit /
          token_set; the function does not check semantic compatibility,
          only shape / device / dtype (via _validate).
    """
    _validate([J_t, J_tp])
    return J_t.float() @ J_tp.float().T
def _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:
    """G = (1/N) * sum_i (J_i @ J_i^T)  (intra-trajectory pool)."""
    U = jacobians[0].shape[0]
    device = jacobians[0].device
    G = torch.zeros(U, U, device=device, dtype=torch.float)
    for J in jacobians:
        G += J.float() @ J.float().T
    G /= len(jacobians)
    return G
def _spectral_decomposition(
    G: torch.Tensor, n_components: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Eigen decomposition, sorted descending."""
    eigenvalues, eigenvectors = torch.linalg.eigh(G)
    # Sort descending
    idx = eigenvalues.argsort(descending=True)
    eigenvalues = eigenvalues[idx][:n_components]
    eigenvectors = eigenvectors[:, idx][:, :n_components].T
    return eigenvalues, eigenvectors


In [ ]:
import warnings
from collections import Counter
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
def uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:
    """Uniform layer mapping from large to small model layers."""
    mapping = {}
    for i in range(n_layers_large):
        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))
        mapping[i] = j
    return mapping
def get_layer_types(model: AutoModelForCausalLM) -> list[str]:
    """Get layer types from model config (full_attention / linear_attention)."""
    cfg = model.config
    return list(
        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types
    )
_DOMAIN_DATASETS = {
    "wiki": ("allenai/c4", "en", "train", "text"),
    "math": ("nvidia/OpenMathInstruct-2", None, "train", "problem"),
    "code": ("neyralabs/apex-agents", None, "train", "prompt"),
    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),
}
def _load_dataset_for_domain(
    domain: str,
) -> tuple[str, str | None, str, str]:
    """Return (dataset_id, config, split, text_field) for a domain.
    Falls back to wiki (C4) if the target dataset is unavailable, with a
    RuntimeWarning: a fallback run silently poisons cross-domain comparisons
    (E-13/E-27 T_domain), so the substitution must be visible in the log.
    """
    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]
    try:
        kw: dict[str, object] = {"split": split, "streaming": True}
        if cfg is not None:
            kw["name"] = cfg
        ds = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
        next(iter(ds))  # force the first fetch: streaming is lazy, a broken/
        # gated dataset would otherwise pass the check and fail inside
        # load_calibration_data (loud crash instead of the intended fallback).
        return ds_id, cfg, split, field
    except Exception:
        warnings.warn(
            f"domain {domain} ({ds_id}) unavailable — falling back to wiki/C4. "
            f"Domain labels in this run are INVALID for cross-domain "
            f"comparisons (E-13/E-27).",
            RuntimeWarning,
            stacklevel=2,
        )
        return _DOMAIN_DATASETS["wiki"]
def _extract_text_from_sample(
    sample: dict,
    field: str,
) -> str:
    """Extract text from a dataset sample, handling different formats."""
    if field == "messages":
        # Chat format: list of {role, content}
        texts = [m.get("content", "") for m in sample.get("messages", [])]
        return " ".join(texts)
    val = sample.get(field, "")
    if isinstance(val, list):
        # e.g. code_contests solutions: list of strings
        return "\n".join(str(v) for v in val)
    if isinstance(val, dict):
        return str(val)
    return val
def load_calibration_data(
    model_name: str,
    max_length: int = 64,
    min_length: int = 64,
    n_samples: int = 4,
    domain: str = "wiki",
    save_offset_mapping: bool = False,
) -> list[dict[str, torch.Tensor]]:
    """Load and tokenize calibration data.
    Defaults match the tested protocol (paper §12/C.6: S=64, filter >= 64,
    N=4); experiments pass explicit lengths. domain: 'wiki' (C4), 'math'
    (OpenMathInstruct-2), 'code' (apex-agents), 'chat' (ultrachat). Falls back
    to wiki if the target dataset is unavailable. save_offset_mapping: adds
    `offset_mapping` to each sample (E-31 sentence boundaries).
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds_id, cfg, split, field = _load_dataset_for_domain(domain)
    kw: dict[str, object] = {"split": split, "streaming": True}
    if cfg is not None:
        kw["name"] = cfg
    dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]
    samples = []
    for raw in dataset:
        text = _extract_text_from_sample(raw, field)
        if not text:
            continue
        if save_offset_mapping:
            toks = tokenizer(
                text, max_length=max_length, truncation=True,
                add_special_tokens=False, return_offsets_mapping=True,
            )
            if len(toks["input_ids"][0]) >= min_length:
                samples.append({
                    "input_ids": toks["input_ids"][0],
                    "offset_mapping": torch.tensor(
                        toks["offset_mapping"][0], dtype=torch.int32
                    ),
                })
        else:
            tokens = tokenizer(
                text, max_length=max_length, truncation=True,
                return_tensors="pt",
            )
            if tokens["input_ids"].shape[1] >= min_length:
                samples.append(tokens)
        if len(samples) >= n_samples:
            break
    return samples
def build_global_token_set(
    model: AutoModelForCausalLM,
    calibration_samples: list[dict[str, torch.Tensor]],
    K: int = 64,
) -> list[int]:
    """Build global token set from top-K logits across all samples."""
    token_counter: Counter[int] = Counter()
    model.eval()  # type: ignore[attr-defined]
    device = next(model.parameters()).device  # type: ignore[attr-defined]
    with torch.no_grad():
        for sample in calibration_samples:
            inputs = {
                "input_ids": sample["input_ids"].to(device),
                "attention_mask": sample["attention_mask"].to(device),
            }
            outputs = model(**inputs)  # type: ignore[operator]
            logits = outputs.logits[:, -1, :]
            top_idx = logits.topk(K, dim=-1).indices[0]
            token_counter.update(top_idx.tolist())
    U = [tid for tid, _ in token_counter.most_common(K)]
    return sorted(U)
def load_models(
    large_name: str, small_name: str, dtype: torch.dtype = torch.float32
) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Load large and small models with eager attention.
    dtype defaults to float32: the tested protocols run on CPU, where fp16
    is ~10x slower (paper §12/C.9). Pass torch.float16 explicitly for GPU.
    Thread count (torch.set_num_threads(4) in §12/C.9) is set by the caller's
    setup, not here.
    """
    kwargs = {
        "torch_dtype": dtype,
        "attn_implementation": "eager",
    }
    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)
    large.eval()  # type: ignore[attr-defined]
    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)
    small.eval()  # type: ignore[attr-defined]
    return large, small
def extract_embeddings(
    model: AutoModelForCausalLM,
) -> torch.Tensor:
    """Extract embedding matrix from model.
    The bridge projects through the AN-embedding matrix W (§3/§5). For the
    tied-weight checkpoints in scope, W = E (embed_tokens). If a checkpoint
    is loaded WITHOUT tied weights, W = lm_head.weight ≠ E and the projection
    would silently use the wrong matrix. We therefore warn ONCE per model if
    lm_head.weight is not the same storage as embed_tokens.weight — a loud,
    non-fatal signal that the W = E assumption does not hold for this model.
    """
    emb = model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]
    _warn_if_not_tied(model, emb)
    return emb
def _warn_if_not_tied(model: AutoModelForCausalLM, emb: torch.Tensor) -> None:
    lm_head = getattr(getattr(model, "lm_head", None), "weight", None)
    if lm_head is None:
        return
    if not lm_head.data_ptr() == emb.data_ptr():
        # data_ptr identity covers tied weights; fall back to value equality
        # for the (rare) case of a tied-but-copied view.
        if not torch.equal(lm_head.detach(), emb):
            warnings.warn(
                f"extract_embeddings: lm_head.weight is NOT tied to "
                f"embed_tokens.weight (model={type(model).__name__}). The "
                f"bridge projects through E assuming W = E (§3); with untied "
                f"weights the an-embedding matrix is lm_head.weight and the "
                f"gap/bridge numbers are silently wrong. Extract lm_head "
                f"explicitly if this is intended.",
                UserWarning,
                stacklevel=2,
            )


In [ ]:
import os
# Max depth under input_root to search (producer puts cache at depth 2:
# results/targets; 4 covers any reasonable layout).
_MAX_DEPTH = 4
def _has_pt(d: str) -> bool:
    try:
        return any(f.endswith(".pt") for f in os.listdir(d))
    except OSError:
        return False
def _walk(root: str, depth: int):
    """Yield dirs under root (listdir-based), depth-limited."""
    try:
        entries = sorted(os.listdir(root))
    except OSError:
        return
    for e in entries:
        p = os.path.join(root, e)
        if _has_pt(p):
            yield p
        if depth > 1:
            yield from _walk(p, depth - 1)
def find_targets_dir(preferred: str, input_root: str = "/kaggle/input") -> str:
    """Return the dir holding the target-cache *.pt files.
    Order: `preferred` (local path / runner-injected), then a listdir walk
    of every mount under `input_root`. Raises FileNotFoundError with the
    mount inventory when nothing is found (no silent fallback, no listdir crash).
    """
    if _has_pt(preferred):
        return preferred
    for hit in _walk(input_root, _MAX_DEPTH):
        return hit
    mounts = []
    try:
        for d in sorted(os.listdir(input_root)):
            try:
                mounts.append(f"{d}/:{os.listdir(os.path.join(input_root, d))[:4]}")
            except OSError:
                mounts.append(f"{d}/:<unreadable>")
    except OSError:
        mounts.append("<no input mount>")
    raise FileNotFoundError(
        f"No target cache .pt in {preferred} nor under {input_root}/ "
        f"(mounts: {mounts}). Run Stage 0 first — it must publish results/targets/*.pt."
    )


In [ ]:
from __future__ import annotations
def _uniform_layer_mapping(n_L: int, n_S: int) -> dict[int, int]:
    """Uniform large->small layer mapping: l_L -> round(l_L*(n_S-1)/(n_L-1)).
    Inlined copy of prism.calibration.uniform_layer_mapping (identical formula)
    so this module stays standalone for kernel inlining.
    """
    return {i: round(i * (n_S - 1) / (n_L - 1)) for i in range(n_L)}
def get_layer_types(model) -> list[str]:
    """Per-layer attention type from model.config.layer_types.
    Qwen3.5 style hybrid: entries 'full_attention' or 'linear_attention'.
    `model.config` (post-AutoModel load) exposes `layer_types` at top level
    (Qwen3_5TextConfig). Returns lowercase list.
    """
    types = getattr(model.config, "layer_types", None)
    if types is None:
        # Fallback: nested text_config (defensive; Qwen3_5 exposes top-level).
        tc = getattr(model.config, "text_config", None)
        types = getattr(tc, "layer_types", None)
    if types is None:
        raise RuntimeError(
            "model.config has no layer_types (and no text_config.layer_types)"
        )
    return [str(t) for t in types]
def select_pairs(n_L: int, n_S: int, layer_types: list[str]) -> list[tuple[int, int]]:
    """Return (l_L, l_S) pairs: full-attn l_S + 0_0 + last, sorted by l_S."""
    if n_L < 1 or n_S < 1 or len(layer_types) != n_S:
        raise ValueError(
            f"invalid: n_L={n_L}, n_S={n_S}, layer_types={len(layer_types)}"
        )
    mapping = _uniform_layer_mapping(n_L, n_S)
    full_S = {i for i, t in enumerate(layer_types) if t == "full_attention"}
    selected = {0, n_S - 1} | full_S
    pairs = [_nearest_preimage(mapping, j) for j in sorted(selected)]
    if (n_L - 1, n_S - 1) not in pairs:
        pairs.append((n_L - 1, n_S - 1))
    return pairs
def _nearest_preimage(mapping: dict[int, int], j: int) -> tuple[int, int]:
    """l_L = argmin |mapping[l] - j|, tie -> smaller l (the selected l_S is j)."""
    best_l, best_d = 0, float("inf")
    for li in range(len(mapping)):
        d = abs(mapping[li] - j)
        if d < best_d:
            best_l, best_d = li, d
    return (best_l, j)
def classify_pair(l_S: int, layer_types: list[str]) -> str:
    """'bridge' if small-model layer l_S has o_proj (full_attention), else 'kl_only'."""
    return (
        "bridge"
        if l_S < len(layer_types) and layer_types[l_S] == "full_attention"
        else "kl_only"
    )


In [ ]:
from __future__ import annotations
import math
import warnings
import torch
__all__ = [
    "top_r_svd",
    "spectrum_from_gramian",
    "eigengap",
    "eigengap_min",
    "entropy_effective_rank",
    "stable_rank",
    "participation_ratio",
    "snr_top1_vs_rest",
    "snr_e14",
    "snr_e14_parts",
    "gauge_share",
    "gauge_share_top1",
]
_LOW_PREC = (torch.float16, torch.bfloat16)
def _check_1d_finite(spectrum: torch.Tensor, name: str = "spectrum") -> None:
    """1-D tensor, no NaN/Inf (index of the first offender in the message)."""
    if spectrum.dim() != 1:
        raise ValueError(f"{name}: expected 1-D tensor, got {spectrum.dim()}-D")
    if spectrum.numel() == 0:
        raise ValueError(f"{name}: empty tensor")
    bad = torch.nonzero(~torch.isfinite(spectrum), as_tuple=False).flatten()
    if bad.numel() > 0:
        raise ValueError(f"{name}: non-finite value at index {int(bad[0])}")
# ---------------------------------------------------------------------------
# SVD primitives
# ---------------------------------------------------------------------------
def top_r_svd(
    J: torch.Tensor, r: int
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Truncated SVD: ``J ≈ U_r @ diag(S_r) @ V_r``.
    Args:
        J: [K, D] Jacobian (float32/float64; fp16/bf16 is cast to float32
           for SVD numerical stability).
        r: 1 ≤ r ≤ min(K, D).
    Returns:
        (U_r [K, r], S_r [r] descending, V_r [r, D]) with the same dtype
        (post-cast) and device as the input.
    Sign caveat: signs of U_r / V_r depend on the LAPACK version — use
    |cos| / Procrustes residuals, never raw sign comparisons.
    """
    if J.dim() != 2:
        raise ValueError(f"top_r_svd: expected 2-D J, got {J.dim()}-D")
    if not torch.isfinite(J).all():
        bad = int((~torch.isfinite(J)).nonzero(as_tuple=False).flatten()[0])
        raise ValueError(
            f"top_r_svd: J contains NaN/Inf (first at flat index {bad}); "
            "refuse to propagate a corrupted cache silently"
        )
    K, D = J.shape
    if not (1 <= r <= min(K, D)):
        raise ValueError(
            f"top_r_svd: r={r} out of range [1, min(K, D)] = [1, {min(K, D)}]"
        )
    if J.dtype in _LOW_PREC:
        J = J.float()
    U, S, V = torch.linalg.svd(J, full_matrices=False)
    return U[:, :r].contiguous(), S[:r].contiguous(), V[:r, :].contiguous()
def spectrum_from_gramian(G: torch.Tensor, ascending: bool = False) -> torch.Tensor:
    """All eigenvalues of a symmetric Gramian, sorted.
    ``eigvalsh`` (no eigenvectors) computed in float64 for stability on
    ill-conditioned ``Δ_U``; the result is cast back to the input dtype.
    Args:
        G: [K, K] symmetric (PSD NOT required: ``Δ_U(t, t')`` for
           ``t ≠ t'`` is symmetric but can have negative eigenvalues).
        ascending: False (default) = descending λ₁ ≥ λ₂ ≥ … (paper §6
           convention); True = ascending (numerical checks).
    Raises:
        ValueError: non-2-D, non-square, or asymmetric (``atol=1e-6``)
           input — no silent symmetrization, that would hide Δ_U bugs.
    """
    if G.dim() != 2 or G.shape[0] != G.shape[1]:
        raise ValueError(f"spectrum_from_gramian: expected square 2-D, got {tuple(G.shape)}")
    if not torch.allclose(G, G.T, atol=1e-6):
        raise ValueError("spectrum_from_gramian: G is not symmetric (no silent symmetrization)")
    lam = torch.linalg.eigvalsh(G.double())
    lam = torch.sort(lam, descending=not ascending).values
    return lam.to(G.dtype)
# ---------------------------------------------------------------------------
# Scalar spectral metrics (take a spectrum, not a matrix)
# ---------------------------------------------------------------------------
def eigengap(spectrum: torch.Tensor) -> float:
    """λ₁ − λ₂ (two largest eigenvalues; paper §6).
    Input order does not matter (sorted internally). Negative eigenvalues
    are NOT dropped: for a proper eigengap ``Δ_U`` should be PSD, but the
    function does not enforce it.
    """
    _check_1d_finite(spectrum)
    if spectrum.numel() < 2:
        raise ValueError(f"eigengap: need K >= 2, got {spectrum.numel()}")
    s = torch.sort(spectrum, descending=True).values
    return float(s[0] - s[1])
def eigengap_min(
    spectrum: torch.Tensor,      # [k] eigenvalues (NOT singular values), descending not required
    eps_frac: float = 0.01,      # threshold fraction of λ_1 = max(spectrum)
) -> tuple[float, bool]:
    """Pre-flight [M-4]: min consecutive eigenvalue gap inside top-k.
    Returns (gap_min, well_separated), well_separated = gap_min >= eps_frac
    * spectrum[0] (after sorting descending; so spectrum[0] is λ_1 = max).
    SEMANTICS (docs/E38_fixes.md §1.1): ``well_separated`` is a
    mathematically neutral predicate — the spectrum is well separated
    (no near-degenerate pair). It is NOT the [M-4] verdict: per [M-4]
    ``min_t gap_min < ε`` (i.e. NOT well_separated) is when the non-Abelian
    formalism is applicable. Callers decide the mapping; E-38c reports
    ``fraction_well_separated`` and drives verdicts from
    ``min_t_gap_min_over_threshold`` directly.
    NOT min(eigengap(...)): eigengap returns only the FIRST gap λ₁−λ₂;
    this returns the MINIMUM over ALL consecutive gaps λ_i − λ_{i+1}.
    Same input convention as eigengap/stable_rank/participation_ratio
    (eigenvalues, i.e. σ²), so it can follow spectrum_from_gramian
    directly without a manual sqrt(). Do NOT pass singular values here.
    The ``>=`` (not ``>``) is deliberate: an exact equality
    ``gap_min == threshold`` is classified as well_separated (boundary =
    "acceptable, not degenerate", per [M-4]).
    Args:
        spectrum: [k] eigenvalues (≥ 0), any order, k ≥ 2.
        eps_frac: threshold as a fraction of λ_1; must lie in (0, 1).
    Returns:
        (gap_min, well_separated) as Python (float, bool).
    Warnings:
        RuntimeWarning: spectrum all zero → returns (0.0, False).
    Errors:
        ValueError: k < 2, NaN/Inf (index reported), negative eigenvalue
            (index reported), eps_frac outside (0, 1).
    """
    _check_1d_finite(spectrum, "spectrum")
    if spectrum.numel() < 2:
        raise ValueError(f"eigengap_min: need k >= 2, got {spectrum.numel()}")
    neg = torch.nonzero(spectrum < 0, as_tuple=False).flatten()
    if neg.numel() > 0:
        raise ValueError(
            f"eigengap_min: negative eigenvalue at index {int(neg[0])} "
            f"(spectrum must be PSD eigenvalues)"
        )
    if not (0.0 < eps_frac < 1.0):
        raise ValueError(f"eigengap_min: eps_frac must be in (0, 1), got {eps_frac}")
    s = torch.sort(spectrum, descending=True).values
    if s[0] == 0:
        warnings.warn("eigengap_min: all eigenvalues are zero", RuntimeWarning)
        return 0.0, False
    # Gaps computed in float64: in float32 (eps ≈ 1.2e-7) a gap near
    # eps_frac * λ_1 with eps_frac ≤ 0.01 would sit at the precision floor
    # (absolute error ~1e-7·λ_1). Same policy as spectrum_from_gramian,
    # which already casts to double internally (API symmetry).
    gaps = (s[:-1] - s[1:]).double()
    gap_min = float(gaps.min())
    threshold = float(eps_frac * s[0])
    return gap_min, gap_min >= threshold
def entropy_effective_rank(spectrum: torch.Tensor) -> float:
    """r_eff = exp(−Σ p_k log p_k), p_k = λ_k / Σλ (paper §6, §15; E-14, E-27.6).
    Non-positive eigenvalues are DROPPED (p_k = 0); if all λ ≤ 0 the
    function returns 0.0 with a RuntimeWarning. For ``Δ_U(t, t')`` with
    t ≠ t' (mixed spectrum) r_eff is computed on the positive part only —
    semantically correct (negative "energy" has no meaning) but the value
    can be smaller than naively expected.
    """
    _check_1d_finite(spectrum)
    pos = spectrum[spectrum > 0]
    if pos.numel() == 0:
        warnings.warn(
            "entropy_effective_rank: all eigenvalues <= 0, returning 0.0",
            RuntimeWarning,
        )
        return 0.0
    p = pos / pos.sum()
    entropy = -(p * torch.log(p)).sum()
    return float(torch.exp(entropy))
def stable_rank(spectrum: torch.Tensor) -> float:
    """Σλ / λ₁ (paper §6).
    "How far the spectrum is from being concentrated in one mode."
    NOT a synonym for participation_ratio: for λ = (4,1,1,1,1),
    stable_rank = 8/4 = 2.0 while PR = 64/20 = 3.2 (TZ said 64/18 ≈ 3.555 — typo, Σλ² = 20).
    Negative eigenvalues are not dropped (ratio definition). λ₁ ≤ 0
    (all zero, or a mixed-sign spectrum where the largest is negative)
    → 0.0 with a RuntimeWarning.
    """
    _check_1d_finite(spectrum)
    s = torch.sort(spectrum, descending=True).values
    lam1 = s[0]
    if lam1 <= 0:
        warnings.warn(
            f"stable_rank: λ₁ = {float(lam1):.3e} <= 0, returning 0.0", RuntimeWarning
        )
        return 0.0
    return float(s.sum() / lam1)
def participation_ratio(spectrum: torch.Tensor) -> float:
    """PR = (Σλ)² / Σλ² ∈ [1, K] (paper §6, §17; E-14).
    Defined on all eigenvalues (negatives not dropped). Σλ² = 0 → 0.0.
    Originally ``_participation_ratio`` in experiments/exp14_snr_pr.py.
    """
    _check_1d_finite(spectrum)
    s2 = spectrum.pow(2).sum()
    if s2 == 0:
        return 0.0
    return float(spectrum.sum().pow(2) / s2)
def snr_top1_vs_rest(spectrum: torch.Tensor) -> float:
    """λ₁ / mean(λ₂..λ_K), λ₁ = largest eigenvalue.
    DEFINITION UNDER REVIEW (TZ docs/new_code_1.md §1.3): the paper §6
    mentions a "leave-one-out SNR" without an exact formula, and the
    original E-14 SNR is sample-based, not spectral (see :func:`snr_e14`).
    This spectral proxy is provided for E-ε; if the original E-14 formula
    turns out to be the one used downstream, prefer ``snr_e14`` and record
    which one is used in ``meta.json``.
    mean(λ_{-1}) = 0 → ``inf``. K = 1 → ValueError. Negatives are not
    dropped. No positive λ → ValueError.
    """
    _check_1d_finite(spectrum)
    if spectrum.numel() < 2:
        raise ValueError(f"snr_top1_vs_rest: need K >= 2, got {spectrum.numel()}")
    s = torch.sort(spectrum, descending=True).values
    lam1 = s[0]
    if lam1 <= 0:
        raise ValueError("snr_top1_vs_rest: no positive eigenvalue (λ₁ <= 0)")
    rest = s[1:].mean()
    if rest == 0:
        return float("inf")
    return float(lam1 / rest)
def snr_e14_parts(
    jacobians: list[torch.Tensor],
) -> tuple[float, float, float]:
    """Original E-14 sample-based SNR with intermediates
    (experiments/exp14_snr_pr.py::``_snr_sample`` — MOVED here, not copied).
    Returns ``(snr, mean_sq, var)`` where
    ``mean_sq = mean‖J‖²_F``, ``var = mean‖J‖²_F − ‖mean J‖²_F`` (per the
    original E-14 definition), and ``snr = ‖mean J‖²_F / max(1e-12, var)``.
    Measures cross-position CONSISTENCY of J, not a spectral property — do
    not confuse with :func:`snr_top1_vs_rest`.
    VALIDATED against results/exp14_snr_pr.json (committed): for all 24
    layers the formula ``(mean_sq − var_sq)/var_sq`` reproduces the stored
    ``SNR`` bit-for-bit (L0 = 0.1415, L21 = 3.6509, L23 = 199.3080 — the
    paper's numbers). See tests/test_spectral.py::test_snr_e14_validated_against_exp14_json.
    """
    if len(jacobians) < 2:
        raise ValueError(f"snr_e14_parts: need >= 2 Jacobians, got {len(jacobians)}")
    J = torch.stack([j.float() for j in jacobians])
    M = J.mean(dim=0)
    signal = M.pow(2).sum().item()
    mean_sq = J.pow(2).sum(dim=(1, 2)).mean().item()
    var = mean_sq - signal
    return float(signal / max(1e-12, var)), float(mean_sq), float(var)
def snr_e14(jacobians: list[torch.Tensor]) -> float:
    """Original E-14 sample-based SNR, scalar form (``snr_e14_parts()[0]``)."""
    return snr_e14_parts(jacobians)[0]
def gauge_share(
    spectrum: torch.Tensor,
    eigenvectors: torch.Tensor,
    uniform: torch.Tensor | None = None,
) -> float:
    """Σ_k (u_kᵀ uniform)² λ_k / Σ_k λ_k (paper §6, "gauge_share ESG-version").
    Fraction of ``JJ^T`` energy lying on the uniform direction in
    token-space.
    Args:
        spectrum: [K] eigenvalues of the Gramian (full spectrum, matching
            ``eigenvectors``).
        eigenvectors: [K, K] with COLUMNS = u_k, the k-th column
            corresponding to ``spectrum[k]``. NOTE: ``esg`` returns
            eigenvectors with ROWS = eigenvectors — pass ``eig.T``.
        uniform: [K] direction; L2-normalized internally. Default
            ``1/√K · ones(K)``.
    The spectrum/eigenvectors pair must cover ALL modes (not top-r) — the
    ``eigenvectors.shape == (K, K)`` check (K = len(spectrum)) enforces this:
    a top-r spectrum paired with a full K×K matrix fails the shape check.
    For top-r use :func:`gauge_share_top1` (or pass the full trace
    explicitly).
    Distinct from ``g_uni`` (paper §8): ``g_uni = ‖QJ‖²_F / ‖J‖²_F`` is
    the energy share of the RAW J; this is the share of ``JJ^T``.
    Σλ = 0 → 0.0 with a RuntimeWarning.
    """
    _check_1d_finite(spectrum)
    K = spectrum.numel()
    if eigenvectors.shape != (K, K):
        raise ValueError(
            f"gauge_share: eigenvectors {tuple(eigenvectors.shape)} "
            f"!= (K, K) = ({K}, {K})"
        )
    if uniform is None:
        uniform = torch.full((K,), 1.0 / math.sqrt(K), device=eigenvectors.device, dtype=eigenvectors.dtype)
    uniform = uniform / uniform.norm()
    coeffs = (eigenvectors.T @ uniform) ** 2  # [K]: (u_k · uniform)^2
    total = spectrum.sum()
    if total == 0:
        warnings.warn("gauge_share: Σλ = 0, returning 0.0", RuntimeWarning)
        return 0.0
    return float((coeffs * spectrum).sum() / total)
def gauge_share_top1(
    spectrum: torch.Tensor, eigenvectors: torch.Tensor
) -> float:
    """λ₁ · (u₁ᵀ uniform)² / Σλ — top-1 proxy (paper §6; E-10a ``gauge_share_0``).
    The value reported in E-10a as ``gauge_share_0`` IS this function
    (top mode only, uniform = 1/√K). ``spectrum[0]`` must be the largest
    eigenvalue (descending convention) and ``eigenvectors[:, 0]`` the
    corresponding eigenvector (columns convention; pass ``eig.T`` from esg).
    Σλ = 0 → 0.0 with a RuntimeWarning.
    """
    _check_1d_finite(spectrum)
    K = spectrum.numel()
    if eigenvectors.shape != (K, K):
        raise ValueError(
            f"gauge_share_top1: eigenvectors {tuple(eigenvectors.shape)} "
            f"!= (K, K) = ({K}, {K})"
        )
    uniform = torch.full((K,), 1.0 / math.sqrt(K), device=eigenvectors.device, dtype=eigenvectors.dtype)
    c1 = (eigenvectors[:, 0] @ uniform) ** 2
    total = spectrum.sum()
    if total == 0:
        warnings.warn("gauge_share_top1: Σλ = 0, returning 0.0", RuntimeWarning)
        return 0.0
    return float(spectrum[0] * c1 / total)


In [ ]:
from __future__ import annotations
import os
import torch
# REPO/JCACHE_DIR are meaningless when inlined into a Kaggle notebook cell
# (no __file__); fall back to cwd — the candidate list in jcache_path() is
# what actually resolves the file there.
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # inlined into a notebook cell
    _HERE = os.getcwd()
REPO = os.path.abspath(os.path.join(_HERE, ".."))
JCACHE_DIR = os.path.join(REPO, "kaggle_kernel", "results")
# Producer kernel slug (kaggle_kernel/push_all.py, exp27b): its kernel-source
# mount does NOT attach on 38d — the producer's outputs are 13 GB, above
# the kernel-source mount limit (observed 2026-10-25: /kaggle/input only
# contained notebooks + self). The cache is therefore ALSO published as two
# Kaggle datasets (same split as exp32r31-tail-recompute uses):
#   jca: L6, L12, L18   /   jcb: L19, L22, L23
# mounted at /kaggle/input/datasets/voroninmichael/prism-exp27b-jcache-{jca,jcb}/.
_E27B_KAGGLE_USERNAME = "voroninmichael"
_E27B_KAGGLE_SLUG = "prism-exp27b-dense-profile"
_E27B_DATASETS = ("prism-exp27b-jcache-jca", "prism-exp27b-jcache-jcb")
LAYERS = (6, 12, 18, 19, 22, 23)
DOMAINS = ("wiki", "math", "code", "chat")
N_PER_DOMAIN = 8
N_DOCS = 32
S = 128
D = 1024
K = 128
def jcache_path(layer: int) -> str:
    """Resolve the per-layer cache file.
    Local: ``kaggle_kernel/results/exp27b_jcache_L{l}.pt``.
    Kaggle: the local path does not exist, so fall back to the producer
    kernel's outputs mount. Returns the canonical local name if neither
    exists (torch.load then fails loudly with a clear path).
    """
    local = os.path.join(JCACHE_DIR, f"exp27b_jcache_L{layer}.pt")
    if os.path.exists(local):
        return local
    # Kaggle: dataset mounts first (reliable), then the kernel-source mount
    # (only attaches if producer outputs are under the mount limit).
    # Dataset mount layout varies by Kaggle version: observed
    # /kaggle/input/<dataset>/ (2026-10-25, v6) and
    # /kaggle/input/datasets/<user>/<dataset>/ (exp32r31 run) — probe both.
    candidates = []
    for ds in _E27B_DATASETS:
        candidates += [
            os.path.join(
                "/kaggle/input", ds, f"exp27b_jcache_L{layer}.pt"
            ),
            os.path.join(
                "/kaggle/input/datasets", _E27B_KAGGLE_USERNAME, ds,
                f"exp27b_jcache_L{layer}.pt",
            ),
        ]
    candidates += [
        os.path.join(
            "/kaggle/input", f"{_E27B_KAGGLE_USERNAME}-{_E27B_KAGGLE_SLUG}",
            "results", f"exp27b_jcache_L{layer}.pt",
        ),
        os.path.join(
            "/kaggle/input", _E27B_KAGGLE_SLUG, "results",
            f"exp27b_jcache_L{layer}.pt",
        ),
    ]
    for mount in candidates:
        if os.path.exists(mount):
            return mount
    # Diagnostic: on Kaggle a miss means the producer mount layout changed
    # or the source wasn't attached — say so in the log loudly.
    if os.path.isdir("/kaggle/input"):
        try:
            lines = []
            for e in sorted(os.listdir("/kaggle/input")):
                p = os.path.join("/kaggle/input", e)
                if os.path.isdir(p):
                    try:
                        sub = sorted(os.listdir(p))[:8]
                    except OSError:
                        sub = ["<unreadable>"]
                    lines.append(f"{e}/ = {sub}")
                else:
                    lines.append(e)
        except OSError:
            lines = ["<unreadable>"]
        print(f"[e38cache] JCACHE MISS for L{layer}; /kaggle/input = {lines}",
              flush=True)
    return local
def load_layer_H(layer: int) -> torch.Tensor:
    """Load ONLY the hidden-state field H from one layer file → [N, S, D].
    For experiments that use H alone (e.g. E-38a): avoids stacking the
    2.1 GB JR tensor (peak ~4.2 GB → ~2.1 GB, and the per-position J/P
    copies are skipped). The file itself is still ~2.1 GB on disk — this
    is a RAM/time saver, not a disk one (docs/E38_fixes2.md §4).
    """
    ck = torch.load(jcache_path(layer), map_location="cpu", weights_only=False)
    if ck["S"] != S or ck["D"] != D:
        raise ValueError(f"layer {layer}: unexpected S={ck['S']} D={ck['D']}")
    n = len(DOMAINS) * N_PER_DOMAIN
    H = torch.empty(n, S, D, dtype=torch.float32)
    i = 0
    for dom in DOMAINS:
        for si in range(N_PER_DOMAIN):
            for t in range(S):
                H[i, t] = ck["H"][(dom, si, t)].float()
            i += 1
    return H
def load_layer(layer: int, want: tuple = ("J", "P", "H")) -> dict:
    """Load one layer file; return dict with stacked arrays.
    ``want`` selects which fields to stack (docs/E38_fixes2.md §6 —
    experiments should only pay for the fields they use): any subset of
    ("J", "P", "H"), default = all three. Skipping H (used by E-38b)
    drops the peak from ~4.2 GB to ~2.1 GB per layer.
    Returns dict with keys (only for requested fields):
        J   torch.Tensor [N_DOCS, S, K, D]  raw J
        P   torch.Tensor [N_DOCS, S, K]     normalized p128
        H   torch.Tensor [N_DOCS, S, D]     hidden states
        U128 list[int]                      token set
    Docs are ordered (domain-major): (wiki,0..7), (math,0..7), (code,0..7),
    (chat,0..7).
    Peak memory is driven by `want`: pass only the fields a consumer uses
    (e.g. want=("P",) for position audits) to stay in the ~2 GB class.
    """
    bad_want = set(want) - {"J", "P", "H"}
    if bad_want:
        raise ValueError(f"load_layer: unknown want fields {sorted(bad_want)}")
    # weights_only=False is deliberate: the cache is an internal artifact we
    # produced ourselves (tensors + tuples + a list of ints), not an
    # untrusted model state dict. Re-evaluate if the cache provenance changes.
    ck = torch.load(jcache_path(layer), map_location="cpu", weights_only=False)
    if ck["S"] != S or ck["D"] != D:
        raise ValueError(f"layer {layer}: unexpected S={ck['S']} D={ck['D']}")
    n = len(DOMAINS) * N_PER_DOMAIN
    J = torch.empty(n, S, K, D, dtype=torch.float32) if "J" in want else None
    P = torch.empty(n, S, K, dtype=torch.float32) if "P" in want else None
    H = torch.empty(n, S, D, dtype=torch.float32) if "H" in want else None
    i = 0
    for dom in DOMAINS:
        for si in range(N_PER_DOMAIN):
            for t in range(S):
                key = (dom, si, t)
                if J is not None:
                    J[i, t] = ck["JR"][key].float()
                if P is not None:
                    P[i, t] = ck["P128"][key].float()
                if H is not None:
                    H[i, t] = ck["H"][key].float()
            i += 1
    out = {}
    if J is not None:
        out["J"] = J
    if P is not None:
        # Guard: gauge_free_J below assumes P is normalized (sum=1). If
        # someone swaps in P128_RAW (sub-stochastic), metrics would
        # silently be "half gauge-free". Fail loudly instead.
        psum = P.sum(dim=-1)
        if not torch.allclose(psum, torch.ones_like(psum), atol=1e-3):
            bad = (psum - 1.0).abs().max().item()
            raise ValueError(f"layer {layer}: P is not normalized (max |sum-1| = {bad:.3e})")
        out["P"] = P
    if H is not None:
        out["H"] = H
    out["U128"] = list(ck["U128"])
    return out
def gauge_free_J(J: torch.Tensor, P: torch.Tensor) -> torch.Tensor:
    """Gauge-free J: (..., K, D) and P (..., K) -> (..., K, D).
    J_gf = p[:, None] * (J - J^T @ p)  — same as exp27_6_isolation.
    Requires P normalized (sum=1 per row): with sub-stochastic P128_RAW the
    result is a silent "half gauge-free" object with different semantics.
    """
    c = torch.einsum("...k,...kd->...d", P, J)  # J^T p, (... , D)
    return P[..., :, None] * (J - c[..., None, :])


In [ ]:
#!/usr/bin/env python3
"""E-38d — Full holonomy matrix H_012 and spectrum {θ_j} (docs/E38.md).
Wave 2 (post E-38c; c reported near_degeneracy on L18/L22 → {θ_j} is
content-bearing, per the pre-registered gate).
For each triple (t, t+1, t+2), each doc, each layer, each R in R_VALUES:
    V_i = top_r_svd(J_i, R)[2].T        [D, R] orthonormal-column basis
    (R_ij, res_ij) = procrustes_align(V_i, V_j)   per-step residual_sq
    H_012 = R_01 @ R_12 @ R_20 (cycle order, float64 — see _procrustes)
    theta = [R//2] spectral angles of H_012 (same binning contract as
            prism.holonomy.holonomy_spectrum; NaN-padded when fewer)
    kappa = ‖H_012 − I‖_F ; trH = tr(H_012)
    Omega = geodesic_curvature(û_t, û_{t+1}, û_{t+2})   (R-independent)
VECTORIZED (v2, 2026-10-24): the old per-triple Python loop was > 2 h on
CPU (126 triples x 32 docs x 6 layers x {3 SVDs + 3 Procrustes + eig}).
v2 computes, per layer, ONE batched SVD over all (doc, t) positions
[N·S, R, D] and three batched [N·(S-2), R, R] Procrustes SVDs, then the
per-triple holonomy H is a small [r, r] matmul loop (cheap). On a T4 this
fits in ~minutes, which frees the 10 T4-h budget for the R extension:
R_VALUES = (16, 32, 64, 128). R = 16 stays the HEADLINE (pre-registered
value; all verdicts use r=16, JSON metrics."16" keeps the old field
names). R > 16 is a POST-HOC r-sweep (sensitivity: does the spectrum
converge with r?) — it does not change any pre-registered outcome.
J = RAW J from the exp27b cache (same convention as E-28, which gave
κ = 0.018 on L23 — consistency check; PRE-REGISTERED tolerance 5e-3,
1e-3 is a tighter DIAGNOSTIC only — the header previously said 1e-3,
contradicting the code; docs/E38_fixes2.md P1-4).
Per layer (headline r=16): {θ_j} distribution over docs, ρ(tr(H), Ω) by
position (Spearman), ‖H−I‖_F vs E-28 κ(L23) = 0.018.
Pre-registered outcomes (per layer, r = 16; docs/E38_fixes2.md P1-5 — the
old docstring did not list them, so the post-hoc ones are now marked):
    consistency_pass / consistency_redflag (L23 only, tol 5e-3)
    effectively_abelian    — max_theta < 0.1  (PRE-REGISTERED; the old
                             spread-based rule mislabeled rigid rotations
                             with identical angles as abelian —
                             docs/E38_fixes2.md P0-1)
    mixed_spectrum         — 0.1 ≤ max_theta ≤ 0.5 (PRE-REGISTERED)
    non_abelian_candidate  — max_theta > 0.5 (POST-HOC extension, no
                             threshold formula in the TZ — recorded as such)
    spherical_geometry     — ρ(trH, Ω) > 0.7 (POST-HOC extension, same)
    third_mechanism        — ρ < 0.3 (POST-HOC extension, same; ≡ NOT
                             spherical, meta.label_semantics)
    partial                — 0.3 ≤ ρ ≤ 0.7 (POST-HOC extension, same)
Degenerate step: a Procrustes step with residual_sq near 2r (≈ 32 for
r=16; subspaces nearly orthogonal, σ_min(V_i^T V_j) ≈ 0) makes R ill-
conditioned. Triples with any such step are EXCLUDED from the ρ
correlation and recorded (per-TZ).
Trivial H_012 ≈ I → spectrum all +1 → theta = [0,...,0]: a VALID Abelian
result, not a bug (recorded as trivial).
Deterministic: seed=0. spearmanr's p is an asymptotic t-approximation
(NOT a permutation test; seed is irrelevant for it) — the median p is
reported as p_med for diagnostics (docs/E38_fixes.md part-5 §3, M-3).
"""
import json
import os
import sys
import time
import numpy as np
import torch
# REPO/OUT are meaningless when inlined into a Kaggle notebook cell
# (no __file__); the Kaggle path is resolved inside main_38dholonomyspectrum() via /kaggle/working.
try:
    _HERE = os.path.abspath(os.path.dirname(__file__))
except NameError:  # inlined into a notebook cell
    _HERE = os.getcwd()
sys.path.insert(0, os.path.abspath(os.path.join(_HERE, "..")))
SEED = 0
R_HEADLINE = 16
R_VALUES = (16, 32, 64, 128)   # v2 extension: r-sweep (budget: T4)
KAPPA_L23_E28 = 0.018
DEGEN_RES_FRAC = 0.9          # degenerate step: residual_sq > frac * 2r
# _HERE is defined above the imports (inlining-safe __file__).
REPO = os.path.abspath(os.path.join(_HERE, ".."))
OUT = os.path.join(REPO, "results", "exp38d_holonomy_spectrum.json")
# OUT is the LOCAL default; main_38dholonomyspectrum() overrides it on Kaggle (see OUT_PATH).
def _procrustes_batched(V_a, V_b):
    """Vectorized procrustes_align (prism/holonomy.py contract).
    V_a, V_b: [B, D, r] orthonormal columns. M = V_b^T V_a [B, r, r]
    (row-major: M_{ij} = Σ_d V_b_{d i} V_a_{d j}), SVD M = P Σ Qt,
    R = Qt^T P^T — EXACTLY the scalar function's construction (minimizes
    ‖V_a R − V_b‖_F). v2.0 had the factor order swapped (V_a^T V_b),
    which gives the same singular values (hence the same residual_sq) but
    a different R — caught by the scalar-vs-batched cross-check.
    Returns (R [B, r, r] float64, residual_sq [B] float):
        residual_sq = 2r − 2·Σ_i s_i, clamped at 0 — EXACTLY the scalar
        formula (‖V_a R − V_b‖_F² = 2r − 2 tr(M^T R) = 2r − 2 Σ s).
    """
    M = torch.bmm(V_b.transpose(1, 2), V_a).double()
    P, s, Qt = torch.linalg.svd(M, full_matrices=False)
    R = torch.bmm(Qt.transpose(1, 2), P.transpose(1, 2))
    res = (2.0 * V_a.shape[2] - 2.0 * s.sum(dim=-1)).clamp(min=0.0)
    return R, res
def _spectrum_batched(H):
    """Vectorized angle set of H: [B, r] float64 orthogonal → angles.
    Mirrors prism/holonomy.holonomy_spectrum's binning contract:
      * one angle per conjugate pair {e^{iθ}, e^{−iθ}}, sorted ascending;
      * +1 axes dropped; the lone-−1 (reflection) axis lands in the π bin;
      * a π pair split across the ±π boundary is merged into one (counted
        once); repeated angles (degenerate spectrum) keep multiplicity;
      * if fewer than r//2 angles survive, the array is NaN-PADDED
        (v1 convention — zero-padding would bias spread high).
    Returns (theta [B, r//2] with NaN padding, n_angles [B]).
    """
    B, r = H.shape[0], H.shape[1]
    Hn = H.numpy()
    half = r // 2
    theta = np.full((B, half), np.nan, dtype=np.float64)
    n_ang = np.zeros(B, dtype=int)
    for bi in range(B):
        angs = np.angle(np.linalg.eigvals(Hn[bi]))   # (-π, π]
        pos = angs[angs > 1e-6]
        if pos.size == 0:
            continue                                  # H ≈ I (or −I? no:
                                                      # −I gives π angles)
        rounded = np.round(pos, 8)
        uniq, counts = np.unique(rounded, return_counts=True)
        mult = counts.copy()
        pi_mask = uniq > np.pi - 1e-6
        mult[pi_mask] //= 2
        mult[mult == 0] = 1   # π-pair split across the ±π boundary
        vals = np.repeat(uniq, mult)
        vals = vals[vals <= np.pi + 1e-12]            # guard: none > π
        vals.sort()
        take = vals[:half]
        theta[bi, :take.size] = take
        n_ang[bi] = take.size
    return theta, n_ang
def main_38dholonomyspectrum(results_dir="results", device="auto"):
    t0 = time.time()
    np.random.seed(SEED)
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"E-38d v2: device={device}, R_VALUES={R_VALUES} (headline {R_HEADLINE})")
    # Output dir: explicit > /kaggle/working > repo (kaggle.md pattern;
    # /kaggle/input is read-only, /kaggle/working survives the run).
    if results_dir:
        out_dir = results_dir
    elif os.path.isdir("/kaggle/working"):
        out_dir = os.path.join("/kaggle/working", "results")
    else:
        out_dir = os.path.join(REPO, "results")
    os.makedirs(out_dir, exist_ok=True)
    OUT_PATH = os.path.join(out_dir, "exp38d_holonomy_spectrum.json")
    from scipy.stats import spearmanr
    def rho_at(trH, omega, degen_mask, n_triples, n_docs):
        """rho_median (+ median p) over non-degenerate finite positions.
        trH/omega/degen_mask are [n_docs, n_triples] (doc=axis0, position=axis1).
        At each fixed position t we correlate trH[:, t] against omega[:, t]
        ACROSS the n_docs=32 documents (the per-position Spearman the TZ wants).
        """
        rs, ps = [], []
        for t in range(0, n_triples, 4):
            m = (~degen_mask[t] & np.isfinite(omega[t])
                 & np.isfinite(trH[t]))
            if m.sum() >= 8:  # P2 (docs/E38_fixes2.md): n_docs=32, 25%
                # survival floor — noisy at 8 points, documented in meta
                r_, p_ = spearmanr(trH[t, m], omega[t, m])
                if np.isfinite(r_):
                    rs.append(float(r_))
                    ps.append(float(p_))
        return (float(np.median(rs)) if rs else float("nan"),
                float(np.median(ps)) if ps else float("nan"), len(rs))
    per_layer = {}
    for l in LAYERS:
        ck = load_layer(l)
        J, H = ck["J"], ck["H"]           # [N, S, K, D], [N, S, D]
        N, Ss, Kk, Dd = J.shape
        n_triples = Ss - 2
        del ck
        norms = H.norm(dim=-1)
        u = H / norms.clamp_min(1e-12)[:, :, None]
        # Ω(t) — R-independent, computed once (vectorized geodesic curvature;
        # same formula as prism.holonomy.geodesic_curvature; NaN where
        # ‖h‖ < 1e-10 or arc < 1e-8 — P1-5).
        u1, u2, u3 = u[:, :-2], u[:, 1:-1], u[:, 2:]
        arc = (torch.norm(u1 - u2, dim=-1) + torch.norm(u3 - u2, dim=-1)) / 2
        t1 = u1 - (u1 * u2).sum(-1, keepdim=True) * u2
        t3 = u3 - (u3 * u2).sum(-1, keepdim=True) * u2
        n1, n3 = torch.norm(t1, dim=-1), torch.norm(t3, dim=-1)
        cos_turn = ((t1 * t3).sum(-1) / (n1 * n3).clamp_min(1e-12)).clamp(-1, 1)
        turn = np.pi - torch.acos(cos_turn)
        # degenerate triple: any of the 3 unit vectors has ‖h‖ < 1e-10
        # (norms is [N, S]; a triple (t, t+1, t+2) sees rows t..t+2)
        deg_h = (torch.stack([norms[:, :-2], norms[:, 1:-1], norms[:, 2:]], dim=0)
                 .min(dim=0).values < 1e-10)
        deg_arc = (arc < 1e-8) | (n1 < 1e-8) | (n3 < 1e-8)
        omega_all = torch.where(deg_h | deg_arc,
                                torch.full_like(arc, float("nan")),
                                turn / arc.clamp_min(1e-8)).numpy().T   # [T, N] (v1 layout)
        n_omega_skipped = int((deg_h | deg_arc).sum())
        per_layer[str(l)] = {}
        # Memory + speed layout (T4 16 GB):
        # ONE fp32 SVD of J per layer (K=128 => full_matrices=False yields
        # all 128 right-singular vectors), so every R in the sweep is a
        # slice of the SAME decomposition — the sweep is strictly
        # comparable (an earlier kernel version did a separate fp64 SVD
        # per R: 4x4096 128x1024 fp64 SVDs per layer ≈ days on T4 fp64 —
        # the kernel was killed at ~5.5 h). fp32 SVD on T4 ≈ 9 min/layer;
        # Vh [N*S,128,D] = 2.15 GB.
        # Per-R peak: Vh 2.15 + J 2.15 + V 2.15 (R=128) + 3 R-blocks fp64
        # 1.59 GB + Htot fp64 0.53 GB ≈ 8.6 GB — comfortably under 16 GB.
        # Procrustes stays fp64 (M [B,R,R] is tiny; T4 fp64 on 4032 small
        # matrices is seconds). Numbers drift ~1e-6 vs the v1 fp64 SVDs
        # (top-16 subspace is gap-protected; E-28 tol is 5e-3).
        Jflat = J.reshape(N * Ss, Kk, Dd)          # [N*S, K, D] fp32
        _, _, Vh = torch.linalg.svd(
            Jflat.to(device, dtype=torch.float32), full_matrices=False)
        del Jflat
        Vt = Vh[:, :R_HEADLINE].to(J.dtype).contiguous().transpose(1, 2)  # [N*S, D, 16]
        for R in R_VALUES:
            if R == R_HEADLINE:
                V = Vt          # [N*S, D, R], shared with the L23 npz block
            else:
                # .contiguous() makes a fresh tensor per R (at most ~2.15 GB
                # at R=128; at R=128 it is a VIEW of Vh — zero extra). The
                # previous R's tensor is GC'd on reassignment. Vh stays
                # alive across the sweep.
                V = Vh[:, :R].to(J.dtype).contiguous().transpose(1, 2)
            # CHUNK OVER DOCUMENTS (v8 OOM fix): the three full-batch V-
            # slices V0/V1/V2 ([N*T, D, R] each = 2.15 GB at R=128) plus the
            # fp64 R-blocks and M transient pushed peak past T4 16 GB.
            # Chunking caps the per-R working set; R=16 gives chunk == N
            # (single pass, bit-identical to the unchunked path).
            per_row = 3 * Dd * R * 4 + 48 * R * R      # 3 V-slices + 48 R-R fp64
            chunk = max(1, min(N, (1500 * 10**6) // (n_triples * per_row) or 1))
            B_tot = N * n_triples
            step_res_cpu = np.empty(B_tot, dtype=np.float64)
            Htot = np.empty((B_tot, R, R), dtype=np.float64)
            for c0 in range(0, N, chunk):
                c1 = min(c0 + chunk, N)
                b0, b1 = c0 * n_triples, c1 * n_triples
                idx = torch.arange(b0, b1, device=device)
                i_doc = idx // n_triples                # flat doc index
                t_pos = idx % n_triples
                f0 = i_doc * Ss + t_pos
                f1 = f0 + 1
                f2 = f0 + 2
                V0, V1, V2 = V[f0], V[f1], V[f2]        # [cT, D, R]
                # R01 (V0,V1) and R20 (V2,V0) first, then V0 is gone before
                # R12 — peak = 2 of the 3 V-slices + 3 R-blocks.
                R01, res01 = _procrustes_batched(V0, V1)
                R20, res20 = _procrustes_batched(V2, V0)
                del V0
                R12, res12 = _procrustes_batched(V1, V2)
                del V1, V2
                step_res_cpu[b0:b1] = torch.maximum(
                    torch.maximum(res01, res12), res20).cpu().numpy()
                # H = R01 @ R12 @ R20, cycle order (v1: R_total = I @ R01
                # @ R12 @ R20, holonomy_accumulate left-multiplication).
                # einsum index check (unit-tested vs torch.bmm chain):
                # R01[b,i,j] R12[b,j,k] R20[b,k,l] -> H[b,i,l]; a transposed
                # index (e.g. 'bsr') silently swaps the last factor.
                Htot[b0:b1] = torch.einsum(
                    "bij,bjk,bkl->bil", R01, R12, R20).cpu().numpy()
                del R01, R12, R20
                if device.startswith("cuda"):
                    torch.cuda.empty_cache()
            if R != R_HEADLINE:
                del V
            degen_flat = (step_res_cpu > DEGEN_RES_FRAC * 2 * R)
            # Layouts (match v1 exactly):
            #   per-TRIPLE stats (kappa, theta, trivial, idiff): flat [B],
            #     B = N*n_triples, index = i_doc*n_triples + t_pos (doc-major).
            #   per-POSITION stats (rho, profile_kappa): [T, N] (v1 layout,
            #     position=axis0, doc=axis1).
            def _tn(flat):
                return flat.reshape(N, n_triples).T
            kappa_flat = np.linalg.norm(Htot - np.eye(R), axis=(1, 2))   # [B]
            trH_all = _tn(np.trace(Htot, axis1=1, axis2=2))              # [T, N]
            theta_flat = _spectrum_batched(torch.tensor(Htot))[0]        # [B, R//2]
            max_theta = np.nanmax(theta_flat, axis=1)                    # [B]
            degen_mask = _tn(degen_flat)                                 # [T, N]
            # exact identity: ||H−I||_F^2 = Σ_j 2(1−cos ang_j) over ALL r
            # eigenvalues (v1's M-2 check, kept per triple)
            angs = np.angle(np.linalg.eigvals(Htot))
            idiff_max = float(np.abs(kappa_flat ** 2
                                     - (2.0 * (1.0 - np.cos(angs))).sum(axis=1)).max())
            trivial = np.all(theta_flat <= 1e-9, axis=1)                 # [B]
            rho_med, p_med, n_rho_pos = rho_at(trH_all, omega_all, degen_mask,
                                               n_triples, N)
            degen_sweep = {}
            for frac in (0.7, 0.9, 0.95):
                msk = _tn(step_res_cpu > frac * 2 * R)
                rm, pm, nm = rho_at(trH_all, omega_all, msk, n_triples, N)
                degen_sweep[f"{frac}x2r"] = {"degenerate_frac": float(msk.mean()),
                                             "rho_median": rm, "n_positions": nm}
            th_masked = theta_flat[~degen_flat]                          # [K, R//2]
            th_vals = th_masked[np.isfinite(th_masked)]
            spread = (float(np.nanmax(th_vals) - np.nanmin(th_vals))
                      if th_vals.size else float("nan"))
            hist = np.histogram(th_vals,
                                bins=[0, 0.01, 0.05, 0.1, 0.5, 1.0, np.pi])[0]
            kappa_tn = _tn(kappa_flat)                                   # [T, N]
            entry = {
                "kappa_median": float(np.median(kappa_flat)),
                "kappa_ci": [float(x) for x in np.percentile(kappa_flat, [2.5, 97.5])],
                "theta": {
                    "median_per_j": [float(np.nanmedian(theta_flat[:, j]))
                                     for j in range(R // 2)],
                    "hist_8_angles": [float(c) for c in hist],
                    "n_dropped_nan": int(th_masked.size - th_vals.size),  # M-4
                    "spread_max_minus_min": spread,
                    "max_theta": float(np.nanmax(max_theta)) if max_theta.size
                    else float("nan"),
                },
                "trH_vs_omega": {"rho_median": rho_med, "n_positions": n_rho_pos,
                                 "p_med": p_med},
                "identity_check_max_absdiff": idiff_max,
                "degenerate_frac": float(degen_flat.mean()),
                "degen_sweep": degen_sweep,
                "trivial_frac": float(trivial.mean()),
                "n_omega_skipped": n_omega_skipped,
                "profile_kappa": [float(np.median(kappa_tn[t]))
                                  for t in range(0, n_triples, 8)],
            }
            per_layer[str(l)][str(R)] = entry
            del Htot
            if device.startswith("cuda"):
                torch.cuda.empty_cache()
        # headline r=16: consistency vs E-28, H_012_doc0, .npz dump
        rh = per_layer[str(l)][str(R_HEADLINE)]
        if l == 23:
            abs_diff = abs(rh["kappa_median"] - KAPPA_L23_E28)
            rh["consistency_e28"] = {
                "kappa_L23_this": rh["kappa_median"],
                "kappa_L23_E28": KAPPA_L23_E28,
                "abs_diff": abs_diff,
                # DIAGNOSTIC, not a hard verdict (docs/E38_fixes.md §1.3):
                # 1e-3 is tighter than the rounding of the cited 0.018 and
                # can false-fail; 5e-3 is the pre-registered tolerance.
                "within_tol_5e-3": bool(abs_diff < 5e-3),
                "within_tol_1e-3": bool(abs_diff < 1e-3)}
        if l == 23:
            # P2 (docs/E38_fixes2.md): full H_012_doc0 as .npz (float64),
            # not JSON. doc 0 = flat indices 0..Ss-1, so the triple
            # (t, t+1, t+2) starts at f0 = t (the old v2 code used
            # f0 = Ss + t — that is doc 1; the .npz was mislabeled).
            f0 = torch.arange(n_triples, device=device)   # doc 0
            V0 = Vt[f0]; V1 = Vt[f0 + 1]; V2 = Vt[f0 + 2]
            R01, _ = _procrustes_batched(V0, V1)
            R12, _ = _procrustes_batched(V1, V2)
            R20, _ = _procrustes_batched(V2, V0)
            H012 = torch.stack(
                [R01[b] @ R12[b] @ R20[b] for b in range(n_triples)], dim=0
            ).cpu().double()
            np.savez(os.path.join(out_dir, f"exp38d_H012_doc0_L{l}.npz"),
                     H012=H012.numpy())
            del V0, V1, V2, R01, R12, R20
            if device.startswith("cuda"):
                torch.cuda.empty_cache()
        r = rh
        del J, H, Vt
        print(f"L{l} r={R_HEADLINE}: κ_med={r['kappa_median']:.5f} "
              f"maxθ={r['theta']['max_theta']:.5f} "
              f"ρ(trH,Ω)={r['trH_vs_omega']['rho_median']:.3f} "
              f"degen={r['degenerate_frac']:.3f} "
              f"trivial={r['trivial_frac']:.3f}")
    # ---- verdicts (headline r=16 ONLY — pre-registered) ----
    verdicts = {}
    for lstr, by_r in per_layer.items():
        r = by_r[str(R_HEADLINE)]
        v = []
        if lstr == "23":
            v.append("consistency_pass" if r["consistency_e28"]["within_tol_5e-3"]
                     else "consistency_redflag")
        sp = r["theta"]["max_theta"]     # P0-1: MAGNITUDE, not spread
        v.append("effectively_abelian" if sp < 0.1
                 else ("non_abelian_candidate" if sp > 0.5 else "mixed_spectrum"))
        rho_m = r["trH_vs_omega"]["rho_median"]
        if np.isfinite(rho_m):
            v.append("spherical_geometry" if rho_m > 0.7
                     else ("third_mechanism" if rho_m < 0.3 else "partial"))
        verdicts[lstr] = ",".join(v)
    out = {
        "params": {"seed": SEED, "r": R_HEADLINE, "r_values": list(R_VALUES),
                   "layers": list(LAYERS), "N_docs": 32,
                   "S": S, "J": "raw (E-28 convention)",
                   "degen_res_threshold": f"{DEGEN_RES_FRAC}*2r",
                   "device": device,
                   "cache": "exp27b_jcache_L{l}.pt"},
        "metrics": per_layer,
        "outcomes": {"per_layer": verdicts,
                     "r_sweep_note": "verdicts use r=16 only (pre-registered); "
                                     "r in (32, 64, 128) is a post-hoc "
                                     "sensitivity sweep (spectrum "
                                     "convergence with r)"},
        "meta": {"label_semantics": "third_mechanism ≡ NOT spherical "
                                    "(ρ < 0.3): the spherical-geometry "
                                    "hypothesis is falsified and the "
                                    "interpretation requires revision — it "
                                    "does NOT assert that a third mechanism "
                                    "was identified (docs/E38_fixes.md "
                                    "part-5 §5)",
                 "note": "v2 vectorized (identical contract to v1's "
                         "per-triple prism.holonomy calls, float64). "
                         "degenerate = any Procrustes step residual_sq > "
                         "0.9*2r (near-orthogonal subspaces); excluded "
                         "from rho. degen_sweep reports degenerate_frac "
                         "and rho_median at 0.7/0.9/0.95*2r (is the 0.9*2r "
                         "cutoff an artifact?). trivial (theta≈0) is a "
                         "valid Abelian result, not a bug. tr(H) and "
                         "‖H−I‖_F² are exact functions of {θ_j} "
                         "(identity_check_max_absdiff). theta_all is "
                         "NaN-padded (mixed spectrum may return < r//2 "
                         "angles); n_dropped_nan counts the histogram "
                         "dropouts. n_omega_skipped = triples with "
                         "‖h‖ < 1e-10 or arc < 1e-8 (Omega undefined, "
                         "excluded from rho). p_med = median spearmanr p "
                         "(asymptotic t-approximation, NOT permutation). "
                         "H_012_doc0 (L23, r=16) is float64 in "
                         "exp38d_H012_doc0_L23.npz. consistency_e28 is a "
                         "diagnostic with an empirical 5e-3 tolerance, "
                         "not a hard verdict (1e-3 is tighter than the "
                         "rounding of the cited 0.018)."},
    }
    def _json_default(o):
        """P0-3 (docs/E38_fixes2.md): strict JSON serializer — raise on
        np.bool_, convert numpy numerics explicitly, NaN/Inf → strings
        (valid RFC-8259)."""
        if isinstance(o, np.bool_):
            raise TypeError(f"unwrapped numpy bool_ in JSON: {o!r}")
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            o = float(o)
        if isinstance(o, float) and not np.isfinite(o):
            return "nan" if o != o else ("inf" if o > 0 else "-inf")
        raise TypeError(
            f"object of type {type(o).__name__} is not JSON serializable")
    with open(OUT_PATH, "w") as f:
        json.dump(out, f, indent=1, default=_json_default)
    print(f"VERDICTS {verdicts}")
    print(f"JSON -> {OUT_PATH} in {time.time() - t0:.1f}s")


In [ ]:
# ---------- run exp38d_holonomy_spectrum ----------
import gc
# Hard cleanup from previous kernel version (same session, different code)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
import inspect
sig = inspect.signature(main_38dholonomyspectrum)
kwargs = {}
params = list(sig.parameters)
if 'small_model' in params: kwargs['small_model'] = SMALL
if 'large_model' in params: kwargs['large_model'] = LARGE
if 'model' in params: kwargs['model'] = SMALL
if 'device_L' in params: kwargs['device_L'] = device_L
if 'device_S' in params: kwargs['device_S'] = device_S
if 'device' in params: kwargs['device'] = device_S
if 'results_dir' in params: kwargs['results_dir'] = RES
if 'targets_dir' in params: kwargs['targets_dir'] = os.path.join(RES, 'targets')
# Remove conflicting keys: if both model and small_model, prefer specific
if 'small_model' in kwargs and 'model' in kwargs: del kwargs['model']
# Memory diagnostic before experiment
if torch.cuda.is_available():
    print(f'cuda:0 alloc={torch.cuda.memory_allocated(0)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(0)/1024**3:.1f}GiB')
    print(f'cuda:1 alloc={torch.cuda.memory_allocated(1)/1024**3:.1f}GiB reserved={torch.cuda.memory_reserved(1)/1024**3:.1f}GiB')
t0 = time.time()
try:
    main_38dholonomyspectrum(**kwargs)
    elapsed = time.time() - t0
    print(f"[OK] exp38d_holonomy_spectrum {elapsed:.0f}s")
    # Persist models (in case experiment loaded from HF but didn't save)
    try:
        dst_s = os.path.join(WORKING, "qwen35-08b")
        dst_l = os.path.join(WORKING, "qwen35-4b")
        if os.path.isdir(dst_s):
            print(f"Small model persisted at {dst_s}")
        if os.path.isdir(dst_l):
            print(f"Large model persisted at {dst_l}")
    except Exception:
        pass
except Exception as e:
    print(f"[FAIL] exp38d_holonomy_spectrum: {e}")
    traceback.print_exc()
print("\nResults:")
if os.path.isdir(RES):
    try:
        for fn in sorted(os.listdir(RES)):
            print(f"  {fn} {os.path.getsize(os.path.join(RES, fn)):,} bytes")
    except OSError:
        pass
